In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Add timezone-aware datetime column (UTC+8) for each if timestamp column exists
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        # convert epoch seconds to timezone-aware datetime in Asia/Shanghai
        df['datetime'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)

# Helper to build compact summary for a dataframe
def build_summary(df, file_label, kpi_col=None):
    rows = df.shape[0]
    # unique cmdb ids sorted, limit to top 50
    if 'cmdb_id' in df.columns:
        cmdb_ids = sorted(df['cmdb_id'].dropna().unique())
        cmdb_ids_trunc = cmdb_ids[:50]
    else:
        cmdb_ids_trunc = []
    # KPI names column handling
    if kpi_col and kpi_col in df.columns:
        kpis = sorted(df[kpi_col].dropna().unique())
        kpis_trunc = kpis[:50]
    elif kpi_col:
        kpis_trunc = []
    else:
        kpis_trunc = None  # indicates no KPI column for this file
    # sample (up to first 10 rows)
    sample = df.head(10).copy()
    # Compact summary as a single-row DataFrame for clean display
    summary = pd.DataFrame([{
        'file': file_label,
        'rows_count': rows,
        'unique_cmdb_ids_count': len(cmdb_ids),
        'unique_cmdb_ids_sample': cmdb_ids_trunc,
        'kpi_field': kpi_col if kpi_col else '(none)',
        'unique_kpis_count': len(kpis) if (kpi_col and kpis_trunc is not None) else None,
        'unique_kpis_sample': kpis_trunc if kpis_trunc is not None else '(field absent)',
        'sample_rows_shown': sample.shape[0]
    }])
    return summary, sample

metric_summary, metric_sample = build_summary(metric_df, 'metric.csv', kpi_col='kpi_name')
trace_summary, trace_sample   = build_summary(trace_df,  'trace.csv',  kpi_col='trace_name')
log_summary, log_sample       = build_summary(log_df,    'log.csv',    kpi_col='log_name')
error_logs_summary, error_logs_sample = build_summary(error_logs_df, 'error_logs.csv', kpi_col=None)

# Display compact results (variables shown, not printed)
metric_summary, metric_sample, trace_summary, trace_sample, log_summary, log_sample, error_logs_summary, error_logs_sample

```
Out[1]:
```
```python
# Build a compact plain-English summary string using previously loaded summary variables
metric_row_count = int(metric_summary.at[0, 'rows_count'])
metric_cmdb_sample = metric_summary.at[0, 'unique_cmdb_ids_sample']
metric_kpi_count = int(metric_summary.at[0, 'unique_kpis_count'])
metric_kpi_sample = metric_summary.at[0, 'unique_kpis_sample'][:10] if isinstance(metric_summary.at[0, 'unique_kpis_sample'], list) else []

trace_row_count = int(trace_summary.at[0, 'rows_count'])
trace_cmdb_sample = trace_summary.at[0, 'unique_cmdb_ids_sample']
trace_kpi_count = int(trace_summary.at[0, 'unique_kpis_count'])
trace_kpi_sample = trace_summary.at[0, 'unique_kpis_sample'][:10] if isinstance(trace_summary.at[0, 'unique_kpis_sample'], list) else []

log_row_count = int(log_summary.at[0, 'rows_count'])
log_cmdb_sample = log_summary.at[0, 'unique_cmdb_ids_sample']
log_kpi_count = int(log_summary.at[0, 'unique_kpis_count'])
log_kpi_sample = log_summary.at[0, 'unique_kpis_sample'][:10] if isinstance(log_summary.at[0, 'unique_kpis_sample'], list) else []

error_row_count = int(error_logs_summary.at[0, 'rows_count'])
error_cmdb_count = int(error_logs_summary.at[0, 'unique_cmdb_ids_count'])

summary_text = (
    f"Summary of telemetry files (compact):\n\n"
    f"1) metric.csv:\n"
    f"   - Total rows: {metric_row_count}\n"
    f"   - Unique components (count): {metric_summary.at[0, 'unique_cmdb_ids_count']}\n"
    f"   - Components sample (sorted, truncated): {metric_cmdb_sample}\n"
    f"   - KPI field: kpi_name (unique KPIs: {metric_kpi_count})\n"
    f"   - KPI sample (first up to 10): {metric_kpi_sample}\n"
    f"   - Sample rows returned: {int(metric_summary.at[0, 'sample_rows_shown'])}\n\n"
    f"2) trace.csv:\n"
    f"   - Total rows: {trace_row_count}\n"
    f"   - Unique components (count): {trace_summary.at[0, 'unique_cmdb_ids_count']}\n"
    f"   - Components sample (sorted, truncated): {trace_cmdb_sample}\n"
    f"   - KPI field: trace_name (unique KPIs: {trace_kpi_count})\n"
    f"   - KPI sample (first up to 10): {trace_kpi_sample}\n"
    f"   - Sample rows returned: {int(trace_summary.at[0, 'sample_rows_shown'])}\n\n"
    f"3) log.csv:\n"
    f"   - Total rows: {log_row_count}\n"
    f"   - Unique components (count): {log_summary.at[0, 'unique_cmdb_ids_count']}\n"
    f"   - Components sample (sorted, truncated): {log_cmdb_sample}\n"
    f"   - KPI field: log_name (unique KPIs: {log_kpi_count})\n"
    f"   - KPI sample (first up to 10): {log_kpi_sample}\n"
    f"   - Sample rows returned: {int(log_summary.at[0, 'sample_rows_shown'])}\n\n"
    f"4) error_logs.csv:\n"
    f"   - Total rows: {error_row_count}\n"
    f"   - Unique components (count): {error_cmdb_count}\n"
    f"   - Fields present include 'message' (no KPI field for this file)\n"
    f"   - Sample rows returned: {int(error_logs_summary.at[0, 'sample_rows_shown'])}\n\n"
    f"Notes: All timestamps were converted to UTC+8 for samples. No threshold or anomaly calculations were performed."
)

# Display the concise summary text
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file  rows_count  unique_cmdb_ids_count                             unique_cmdb_ids_sample kpi_field  unique_kpis_count                                 unique_kpis_sample  sample_rows_shown
0  metric.csv       27536                     29  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...  kpi_name                359  [Container-DOCKER_CONTAINER_1bc4fc80d241--bcou...                 10,     timestamp cmdb_id                                     kpi_name         value                  datetime
0  1615271400    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09 2021-03-09 06:30:00+08:00
1  1615271400    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  2.836170e+01 2021-03-09 06:30:00+08:00
2  1615271400    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.705202e+08 2021-03-09 06:30:00+08:00
3  1615271400    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.791450e+07 2021-03-09 06:30:00+08:00
4  1615271400    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad  4.822000e-01 2021-03-09 06:30:00+08:00
5  1615271400    IG01              JVM-Runtime_7778_JVM_JVM_Uptime  3.504450e+05 2021-03-09 06:30:00+08:00
6  1615271400    IG01     JVM-Threads_7778_JVM_ThreadCount_Threads  7.200000e+01 2021-03-09 06:30:00+08:00
7  1615271400    IG01              OSLinux-CPU_CPU-0_SingleCpuUtil  1.729600e+00 2021-03-09 06:30:00+08:00
8  1615271400    IG01              OSLinux-CPU_CPU-0_SingleCpuidle  9.878495e+01 2021-03-09 06:30:00+08:00
9  1615271400    IG01              OSLinux-CPU_CPU-2_SingleCpuUtil  1.382200e+00 2021-03-09 06:30:00+08:00,         file  rows_count  unique_cmdb_ids_count                             unique_cmdb_ids_sample   kpi_field  unique_kpis_count                                 unique_kpis_sample  sample_rows_shown
0  trace.csv        7125                     12  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...  trace_name                 72  [trace.from_IG01.duration_mean, trace.from_IG0...                 10,     timestamp cmdb_id                       trace_name        value                  datetime
0  1615271400    IG01    trace.from_root.duration_mean     0.566750 2021-03-09 06:30:00+08:00
1  1615271400    IG01     trace.from_root.duration_p95     1.810550 2021-03-09 06:30:00+08:00
2  1615271400    IG01        trace.from_root.row_count     4.000000 2021-03-09 06:30:00+08:00
3  1615271400    IG01         trace.self.duration_mean     0.645663 2021-03-09 06:30:00+08:00
4  1615271400    IG01          trace.self.duration_p95     2.645600 2021-03-09 06:30:00+08:00
5  1615271400    IG01             trace.self.row_count  1957.000000 2021-03-09 06:30:00+08:00
6  1615271400    IG01  trace.to_Tomcat01.duration_mean     0.628626 2021-03-09 06:30:00+08:00
7  1615271400    IG01   trace.to_Tomcat01.duration_p95     3.072700 2021-03-09 06:30:00+08:00
8  1615271400    IG01      trace.to_Tomcat01.row_count   230.000000 2021-03-09 06:30:00+08:00
9  1615271400    IG01  trace.to_Tomcat02.duration_mean     0.672174 2021-03-09 06:30:00+08:00,       file  rows_count  unique_cmdb_ids_count                             unique_cmdb_ids_sample kpi_field  unique_kpis_count                                 unique_kpis_sample  sample_rows_shown
0  log.csv         506                      7  [Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat...  log_name                  3  [log.error_count, log.gc_pause_max, log.row_co...                 10,     timestamp   cmdb_id          log_name   value                  datetime
0  1615271400   Mysql01   log.error_count    0.00 2021-03-09 06:30:00+08:00
1  1615271400   Mysql01     log.row_count    4.00 2021-03-09 06:30:00+08:00
2  1615271400  Tomcat01   log.error_count    0.00 2021-03-09 06:30:00+08:00
3  1615271400  Tomcat01     log.row_count  504.00 2021-03-09 06:30:00+08:00
4  1615271400  Tomcat03   log.error_count    0.00 2021-03-09 06:30:00+08:00
5  1615271400  Tomcat03     log.row_count  513.00 2021-03-09 06:30:00+08:00
6  1615271400  Tomcat04   log.error_count    0.00 2021-03-09 06:30:00+08:00
7  1615271400  Tomcat04  log.gc_pause_max    0.07 2021-03-09 06:30:00+08:00
8  1615271400  Tomcat04     log.row_count  522.00 2021-03-09 06:30:00+08:00
9  1615271400  apache01   log.error_count    0.00 2021-03-09 06:30:00+08:00,              file  rows_count  unique_cmdb_ids_count unique_cmdb_ids_sample kpi_field unique_kpis_count unique_kpis_sample  sample_rows_shown
0  error_logs.csv           0                      0                     []    (none)              None     (field absent)                  0, Empty DataFrame
Columns: [timestamp, cmdb_id, message, datetime]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timestamp and value types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# 1) Define incident window using timezone-aware conversion (numeric epoch seconds)
start_dt = tz.localize(datetime(2021, 3, 9, 14, 30, 0))
end_dt   = tz.localize(datetime(2021, 3, 9, 15,  0, 0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

# 2) Candidate components list (from provided possible root cause components)
candidates = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# 3) Compute global percentiles and aggregates for each (cmdb_id, kpi_name) using full series
metrics_candidates_df = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# Remove rows with non-numeric values
metrics_candidates_df = metrics_candidates_df.dropna(subset=['value'])

# Basic agg: count, min, max, mean
agg_basic = metrics_candidates_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    count_all='count', min_all='min', max_all='max', mean_all='mean'
)

# Quantiles: P95, P90, P50, P5 (computed on full series BEFORE any window filtering)
quantiles = metrics_candidates_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(
    [0.95, 0.90, 0.50, 0.05]
).unstack(level=-1).rename(columns={0.95: 'P95', 0.90: 'P90', 0.50: 'P50', 0.05: 'P5'})

# Combine into thresholds (global)
thresholds = agg_basic.join(quantiles).reset_index()

# 4) Filter metric.csv to rows within the incident window
window_df = metrics_candidates_df[(metrics_candidates_df['timestamp'] >= start_ts) &
                                  (metrics_candidates_df['timestamp'] <= end_ts)].copy()

# 5) For each candidate compute within-window aggregates and anomalies using global P95
# Window basic stats
window_basic = window_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    total_points_in_window='count', max_value_in_window='max'
).reset_index()

# Attach global thresholds (P95,P5) to window rows to evaluate anomalies
window_with_thresh = window_df.merge(thresholds[['cmdb_id', 'kpi_name', 'P95', 'P5']],
                                     on=['cmdb_id', 'kpi_name'],
                                     how='left')

# Mark anomalies where value >= global P95
window_with_thresh['is_anomaly'] = window_with_thresh['value'] >= window_with_thresh['P95']

# Compute anomaly counts and earliest anomaly timestamp per (cmdb_id,kpi_name)
anom_agg = window_with_thresh[window_with_thresh['is_anomaly']].groupby(
    ['cmdb_id', 'kpi_name']
).agg(
    anomaly_count_using_P95=('is_anomaly', 'sum'),
    earliest_anomaly_timestamp_in_window=('timestamp', 'min')
).reset_index()

# Merge window_basic with anomaly aggregates and global thresholds
anomaly_candidates = window_basic.merge(anom_agg, on=['cmdb_id', 'kpi_name'], how='left')
anomaly_candidates = anomaly_candidates.merge(thresholds[['cmdb_id', 'kpi_name', 'P95', 'P5']],
                                              on=['cmdb_id', 'kpi_name'], how='left')

# Fill NaN anomaly counts with 0
anomaly_candidates['anomaly_count_using_P95'] = anomaly_candidates['anomaly_count_using_P95'].fillna(0).astype(int)
# earliest_anomaly_timestamp_in_window keep as float/NaN (no conversion here), convert to Int where present
anomaly_candidates['earliest_anomaly_timestamp_in_window'] = anomaly_candidates['earliest_anomaly_timestamp_in_window'].where(
    ~anomaly_candidates['earliest_anomaly_timestamp_in_window'].isna(),
    None
)

# 6A) thresholds table (compact): cmdb_id, kpi_name, P95, P5, count_all sorted by count_all desc, top 50
thresholds_df = thresholds.rename(columns={'count_all': 'count_all'})[
    ['cmdb_id', 'kpi_name', 'P95', 'P5', 'count_all']
].sort_values(by='count_all', ascending=False).head(50).reset_index(drop=True)

# 6B) anomaly candidates in window: select and sort by anomaly_count desc then max_value desc, top 20
anomalies_df = anomaly_candidates[[
    'cmdb_id', 'kpi_name', 'total_points_in_window', 'anomaly_count_using_P95',
    'earliest_anomaly_timestamp_in_window', 'max_value_in_window', 'P95', 'P5'
]].sort_values(by=['anomaly_count_using_P95', 'max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# Keep outputs compact: show numeric start_ts and end_ts plus the two result DataFrames
start_ts, end_ts, thresholds_df, anomalies_df

```
Out[2]:
```
```python
# Build a compact plain-English summary using cached variables from prior analysis
import pytz
from datetime import datetime

# Reuse existing variables if present
try:
    start_ts
except NameError:
    start_ts = 1615271400
try:
    end_ts
except NameError:
    end_ts = 1615273200

# Prefer full thresholds if available, otherwise use thresholds_df (top-50)
try:
    thresholds_full
    thresholds_source = thresholds_full
except NameError:
    thresholds_source = thresholds_df

# Use anomalies_df from kernel
anoms = anomalies_df

# Candidate components list used
components_used = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                   "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Compact counts
total_candidate_pairs = int(thresholds_source.shape[0])  # note: thresholds_df was limited to top50 earlier
anomaly_candidates_count = int(anoms.shape[0])

# Prepare top examples (truncate long KPI names for compactness)
def short(s, n=80):
    s = str(s)
    return (s if len(s)<=n else s[:n-3]+'...')

top_thresholds = []
for _, r in thresholds_source.head(5).iterrows():
    top_thresholds.append({
        'cmdb_id': r['cmdb_id'],
        'kpi_name': short(r['kpi_name'], 80),
        'count_all': int(r['count_all']),
        'P95': float(r['P95']),
        'P5': float(r['P5'])
    })

top_anoms = []
for _, r in anoms.head(5).iterrows():
    top_anoms.append({
        'cmdb_id': r['cmdb_id'],
        'kpi_name': short(r['kpi_name'], 80),
        'total_points_in_window': int(r['total_points_in_window']),
        'anomaly_count_using_P95': int(r['anomaly_count_using_P95']),
        'earliest_anomaly_ts': r['earliest_anomaly_timestamp_in_window'],
        'max_value_in_window': float(r['max_value_in_window']),
        'global_P95': float(r['P95']),
        'global_P5': float(r['P5'])
    })

# Convert numeric window endpoints to readable UTC+8 datetimes for context
tz = pytz.timezone('Asia/Shanghai')
start_dt_str = datetime.fromtimestamp(int(start_ts), tz).strftime('%Y-%m-%d %H:%M:%S%z')
end_dt_str = datetime.fromtimestamp(int(end_ts), tz).strftime('%Y-%m-%d %H:%M:%S%z')

summary_text = (
    "Incident window numeric timestamps used:\n"
    f"  - start_ts = {int(start_ts)}  (UTC+8: {start_dt_str})\n"
    f"  - end_ts   = {int(end_ts)}  (UTC+8: {end_dt_str})\n\n"
    "Components considered (filtered to provided candidate list).\n\n"
    f"Thresholds summary (global percentiles computed on full series):\n"
    f"  - Candidate (cmdb_id, kpi_name) pairs considered (shown table rows): {total_candidate_pairs}\n"
    "  - Top 5 threshold rows (cmdb_id, kpi_name, count_all, P95, P5):\n"
)
for t in top_thresholds:
    summary_text += f"    - {t['cmdb_id']}, {t['kpi_name']}, count={t['count_all']}, P95={t['P95']:.6g}, P5={t['P5']:.6g}\n"

summary_text += (
    "\nAnomaly candidates inside the incident window (start_ts <= timestamp <= end_ts):\n"
    f"  - Number of (cmdb_id,kpi_name) with any window data (rows shown): {anomaly_candidates_count}\n"
    "  - Top 5 anomaly candidates (cmdb_id, kpi_name, total_points_in_window, anomaly_count_using_P95, "
    "earliest_anomaly_ts, max_value_in_window, global_P95, global_P5):\n"
)
for a in top_anoms:
    earliest = a['earliest_anomaly_ts'] if a['earliest_anomaly_ts'] is not None else 'N/A'
    summary_text += (
        f"    - {a['cmdb_id']}, {a['kpi_name']}, pts={a['total_points_in_window']}, "
        f"anom_cnt={a['anomaly_count_using_P95']}, earliest={earliest}, max={a['max_value_in_window']:.6g}, "
        f"P95={a['global_P95']:.6g}, P5={a['global_P5']:.6g}\n"
    )

summary_text += (
    "\nNotes:\n"
    "  - Global percentiles (P95/P5 etc.) were computed on the FULL historical series per (cmdb_id,kpi_name) before filtering to the incident window.\n"
    "  - Thresholds table was limited to top 50 rows by overall count (as requested); anomaly candidates limited to top 20 when returned earlier (we show top 5 examples here).\n"
)

# Display the compact plain-English summary string
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(1615271400, 1615273200,      cmdb_id                                           kpi_name           P95            P5  count_all
0       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads  7.450000e+01  6.700000e+01         31
1       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.791450e+07  5.791450e+07         31
2       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09         31
3       IG01                        OSLinux-CPU_CPU_CPUidleutil  7.409403e+01  7.355420e+01         31
4   apache02                         OSLinux-CPU_CPU_CPUSysTime  4.800405e-01  2.576500e-01         31
..       ...                                                ...           ...           ...        ...
45  Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter  0.000000e+00  0.000000e+00         31
46  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter  0.000000e+00  0.000000e+00         31
47  Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions  0.000000e+00  0.000000e+00         31
48  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  0.000000e+00  0.000000e+00         31
49  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  0.000000e+00  0.000000e+00         31

[50 rows x 5 columns],      cmdb_id                                           kpi_name  total_points_in_window  anomaly_count_using_P95  earliest_anomaly_timestamp_in_window  max_value_in_window           P95            P5
0       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                      31                       31                            1615271400         2.040136e+09  2.040136e+09  2.040136e+09
1       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                      31                       31                            1615271400         2.040136e+09  2.040136e+09  2.040136e+09
2       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                      31                       31                            1615271400         2.040136e+09  2.040136e+09  2.040136e+09
3       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                      31                       31                            1615271400         2.040136e+09  2.040136e+09  2.040136e+09
4       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                      31                       31                            1615271400         9.471166e+08  9.471166e+08  9.471166e+08
5       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                      31                       31                            1615271400         5.069049e+08  5.069049e+08  5.069049e+08
6       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                      31                       31                            1615271400         5.791450e+07  5.791450e+07  5.791450e+07
7   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...                      31                       31                            1615271400         4.053960e+05  4.053960e+05  4.053960e+05
8   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...                      31                       31                            1615271400         3.303510e+05  3.303510e+05  3.303510e+05
9   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...                      31                       31                            1615271400         3.300540e+05  3.300540e+05  3.300540e+05
10  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...                      31                       31                            1615271400         3.248520e+05  3.248520e+05  3.248520e+05
11  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...                      31                       31                            1615271400         9.250000e+02  9.250000e+02  9.250000e+02
12  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...                      31                       31                            1615271400         5.000000e+02  5.000000e+02  5.000000e+02
13  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...                      31                       31                            1615271400         5.000000e+02  5.000000e+02  5.000000e+02
14  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...                      31                       31                            1615271400         5.000000e+02  5.000000e+02  5.000000e+02
15  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...                      31                       31                            1615271400         5.000000e+02  5.000000e+02  5.000000e+02
16  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...                      31                       31                            1615271400         2.960000e+02  2.960000e+02  2.960000e+02
17  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...                      31                       31                            1615271400         2.700000e+02  2.700000e+02  2.700000e+02
18  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...                      31                       31                            1615271400         2.290000e+02  2.290000e+02  2.290000e+02
19   Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                      31                       31                            1615271400         2.400000e+01  2.400000e+01  2.400000e+01)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached variables: metric_df, thresholds_df, anomalies_df, window_df, start_ts, end_ts
# Ensure required variables exist
for v in ['metric_df', 'thresholds_df', 'anomalies_df', 'window_df', 'start_ts', 'end_ts']:
    if v not in globals():
        raise NameError(f"Required cached variable '{v}' is missing.")

# Prepare a lookup of global thresholds (P95,P5) indexed by (cmdb_id,kpi_name)
thr_lookup = thresholds_df.set_index(['cmdb_id','kpi_name'])[['P95','P5']]

# Candidate pairs to evaluate: those present in anomalies_df
pairs_df = anomalies_df[['cmdb_id','kpi_name','total_points_in_window']].drop_duplicates().reset_index(drop=True)

# Container for fault segment records
segments = []

# Iterate pairs and scan window rows for consecutive runs where value >= global P95
for _, row in pairs_df.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    total_pts = int(row['total_points_in_window'])
    # Get P95,P5 from thresholds lookup; if missing, skip this pair
    try:
        P95 = float(thr_lookup.at[(cmdb,kpi),'P95'])
        P5  = float(thr_lookup.at[(cmdb,kpi),'P5'])
    except KeyError:
        # no global threshold available; skip
        continue

    # Extract window rows for this pair ordered by timestamp
    sub = window_df[(window_df['cmdb_id']==cmdb) & (window_df['kpi_name']==kpi)].copy()
    if sub.empty:
        continue
    sub = sub.sort_values('timestamp').reset_index(drop=True)

    # Boolean mask where value >= P95
    sub['is_above'] = sub['value'] >= P95

    # Scan rows to find consecutive runs of is_above True with contiguous minute timestamps (diff == 60)
    run_active = False
    run_start_ts = None
    run_end_ts = None
    run_max = -np.inf
    run_count = 0
    prev_ts = None

    for i, r in sub.iterrows():
        ts = int(r['timestamp'])
        val = float(r['value'])
        is_above = bool(r['is_above'])

        if is_above:
            if not run_active:
                # start new run
                run_active = True
                run_start_ts = ts
                run_end_ts = ts
                run_max = val
                run_count = 1
            else:
                # check contiguous minute
                if prev_ts is not None and ts - prev_ts == 60:
                    # continue run
                    run_end_ts = ts
                    run_count += 1
                    if val > run_max:
                        run_max = val
                else:
                    # gap in time -> close previous run and start new
                    # append previous run
                    segments.append({
                        'cmdb_id': cmdb,
                        'kpi_name': kpi,
                        'fault_start_ts': run_start_ts,
                        'fault_end_ts': run_end_ts,
                        'duration_points': run_count,
                        'anomaly_count': run_count,
                        'total_points_in_window': total_pts,
                        'max_value_in_segment': run_max,
                        'global_P95': P95,
                        'global_P5': P5
                    })
                    # start new run
                    run_start_ts = ts
                    run_end_ts = ts
                    run_max = val
                    run_count = 1
            # set prev_ts
            prev_ts = ts
        else:
            # if currently in a run, close it
            if run_active:
                segments.append({
                    'cmdb_id': cmdb,
                    'kpi_name': kpi,
                    'fault_start_ts': run_start_ts,
                    'fault_end_ts': run_end_ts,
                    'duration_points': run_count,
                    'anomaly_count': run_count,
                    'total_points_in_window': total_pts,
                    'max_value_in_segment': run_max,
                    'global_P95': P95,
                    'global_P5': P5
                })
                run_active = False
                run_start_ts = run_end_ts = None
                run_max = -np.inf
                run_count = 0
            prev_ts = ts

    # after loop, if still active close
    if run_active:
        segments.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'fault_start_ts': run_start_ts,
            'fault_end_ts': run_end_ts,
            'duration_points': run_count,
            'anomaly_count': run_count,
            'total_points_in_window': total_pts,
            'max_value_in_segment': run_max,
            'global_P95': P95,
            'global_P5': P5
        })

# Convert segments to DataFrame (cache intermediate)
fault_segments = pd.DataFrame(segments)

# If no segments found, create empty DF with required columns
required_cols = ['cmdb_id','kpi_name','fault_start_ts','fault_end_ts','duration_points',
                 'anomaly_count','total_points_in_window','max_value_in_segment','global_P95','global_P5']
if fault_segments.empty:
    fault_segments = pd.DataFrame(columns=required_cols)

# Compute breach_ratio safely
def compute_breach(row):
    P95 = row['global_P95']
    maxv = row['max_value_in_segment']
    if pd.isna(P95) or P95 == 0:
        if maxv > 0:
            return float('inf')
        else:
            return 0.0
    else:
        return (maxv - P95) / P95

if not fault_segments.empty:
    fault_segments['breach_ratio'] = fault_segments.apply(compute_breach, axis=1)
else:
    fault_segments['breach_ratio'] = pd.Series(dtype=float)

# 2) Filter fault segments to likely real faults:
# Keep segments where breach_ratio > 0.5 OR anomaly_count >= max(2, 0.5 * total_points_in_window)
if not fault_segments.empty:
    # threshold count per segment
    fault_segments['min_required_anoms'] = fault_segments['total_points_in_window'].apply(lambda tp: max(2, 0.5 * tp))
    filtered_faults = fault_segments[
        (fault_segments['breach_ratio'] > 0.5) |
        (fault_segments['anomaly_count'] >= fault_segments['min_required_anoms'])
    ].copy()
else:
    filtered_faults = fault_segments.copy()

# Keep only requested columns for output and order them
output_cols = ['cmdb_id','kpi_name','fault_start_ts','fault_end_ts','duration_points',
               'anomaly_count','total_points_in_window','max_value_in_segment','global_P95','global_P5','breach_ratio']
if not filtered_faults.empty:
    filtered_faults = filtered_faults[output_cols]
    # sort by breach_ratio desc then anomaly_count desc
    filtered_faults = filtered_faults.sort_values(by=['breach_ratio','anomaly_count'], ascending=[False, False]).reset_index(drop=True)
    # limit to top 20 rows for output
    filtered_faults_top20 = filtered_faults.head(20).copy()
    # earliest fault_start_ts among returned faults
    earliest_fault_start_ts = int(filtered_faults_top20['fault_start_ts'].min()) if not filtered_faults_top20.empty else None
else:
    filtered_faults_top20 = filtered_faults.copy()
    earliest_fault_start_ts = None

# 3) Compact result: one-line summary and faults table (top 20)
total_evaluated_pairs = int(pairs_df.shape[0])
num_fault_segments_after_filter = int(filtered_faults.shape[0])

summary_line = (
    f"Evaluated candidate pairs: {total_evaluated_pairs}; "
    f"Fault segments after filtering: {num_fault_segments_after_filter}."
)

# Display compact outputs (variables only)
summary_line, filtered_faults_top20, earliest_fault_start_ts

```
Out[3]:
```
```python
# Build a compact plain-English summary string using cached results
import pytz
from datetime import datetime

# Reuse cached variables
try:
    summary_line
    filtered_faults_top20
    earliest_fault_start_ts
except NameError:
    # Fallback if not available
    summary_line = "Evaluated candidate pairs: N/A; Fault segments after filtering: N/A."
    filtered_faults_top20 = None
    earliest_fault_start_ts = None

tz = pytz.timezone('Asia/Shanghai')

def ts_to_str(ts):
    if ts is None:
        return 'null'
    return datetime.fromtimestamp(int(ts), tz).strftime('%Y-%m-%d %H:%M:%S%z')

def short(s, n=80):
    s = str(s)
    return s if len(s)<=n else s[:n-3]+'...'

# Compose summary
lines = []
lines.append("Summary of fault-segment scan (plain English):")
lines.append(summary_line)
lines.append(f"Earliest fault_start_ts among returned faults: {earliest_fault_start_ts} (UTC+8: {ts_to_str(earliest_fault_start_ts)})")
lines.append("")
lines.append("Filtered fault segments (up to 20 rows) — columns: cmdb_id, kpi_name, fault_start_ts, fault_end_ts, duration_points, anomaly_count, total_points_in_window, max_value_in_segment, global_P95, global_P5, breach_ratio:")
if filtered_faults_top20 is None or filtered_faults_top20.empty:
    lines.append("  - None")
else:
    for _, r in filtered_faults_top20.iterrows():
        lines.append(
            f"  - {r['cmdb_id']}, {short(r['kpi_name'],80)}, "
            f"start={int(r['fault_start_ts'])}({ts_to_str(r['fault_start_ts'])}), "
            f"end={int(r['fault_end_ts'])}({ts_to_str(r['fault_end_ts'])}), "
            f"duration_pts={int(r['duration_points'])}, anom_cnt={int(r['anomaly_count'])}, "
            f"total_pts={int(r['total_points_in_window'])}, max={float(r['max_value_in_segment']):.6g}, "
            f"P95={float(r['global_P95']):.6g}, P5={float(r['global_P5']):.6g}, breach_ratio={float(r['breach_ratio']):.6g}"
        )

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

('Evaluated candidate pairs: 20; Fault segments after filtering: 7.',     cmdb_id                                           kpi_name  fault_start_ts  fault_end_ts  duration_points  anomaly_count  total_points_in_window  max_value_in_segment    global_P95     global_P5  breach_ratio
0      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax      1615271400    1615273200               31             31                      31          2.040136e+09  2.040136e+09  2.040136e+09           0.0
1      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax      1615271400    1615273200               31             31                      31          2.040136e+09  2.040136e+09  2.040136e+09           0.0
2      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed      1615271400    1615273200               31             31                      31          5.791450e+07  5.791450e+07  5.791450e+07           0.0
3  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...      1615271400    1615273200               31             31                      31          3.303510e+05  3.303510e+05  3.303510e+05           0.0
4  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...      1615271400    1615273200               31             31                      31          3.248520e+05  3.248520e+05  3.248520e+05           0.0
5  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...      1615271400    1615273200               31             31                      31          2.700000e+02  2.700000e+02  2.700000e+02           0.0
6   Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount      1615271400    1615273200               31             31                      31          2.400000e+01  2.400000e+01  2.400000e+01           0.0, 1615271400)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached variables when available
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure timestamps and values are proper types
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Reuse incident window numeric timestamps
try:
    start_ts, end_ts
except NameError:
    # fallback (shouldn't happen since cached), but keep safe defaults
    start_ts = 1615271400
    end_ts = 1615273200

# 1) Determine component set from cached filtered faults (filtered_faults_top20) or fallback list
try:
    comp_set = sorted(filtered_faults_top20['cmdb_id'].dropna().unique().tolist())
    if len(comp_set) == 0:
        raise KeyError
except Exception:
    comp_set = ["IG01","IG02","Tomcat03","Tomcat04","Mysql02"]

# 2) Compute global percentiles and aggregates for (cmdb_id, trace_name) on full trace series
trace_candidates = trace_df[trace_df['cmdb_id'].isin(comp_set)].copy()
trace_candidates = trace_candidates.dropna(subset=['value'])

agg_basic = trace_candidates.groupby(['cmdb_id','trace_name'])['value'].agg(
    count_all='count', min_all='min', max_all='max', mean_all='mean'
)

quantiles = trace_candidates.groupby(['cmdb_id','trace_name'])['value'].quantile(
    [0.95, 0.90, 0.50, 0.05]
).unstack(level=-1).rename(columns={0.95:'P95', 0.90:'P90', 0.50:'P50', 0.05:'P5'})

trace_thresholds = agg_basic.join(quantiles).reset_index()

# 3) Filter traces to incident window and compute within-window anomaly aggregates using global P95
window_traces = trace_candidates[(trace_candidates['timestamp'] >= int(start_ts)) &
                                 (trace_candidates['timestamp'] <= int(end_ts))].copy()

# Merge thresholds into window rows to evaluate anomalies
window_with_thresh = window_traces.merge(
    trace_thresholds[['cmdb_id','trace_name','P95','P5']],
    on=['cmdb_id','trace_name'],
    how='left'
)

window_with_thresh['is_anomaly'] = window_with_thresh['value'] >= window_with_thresh['P95']

# Window basic stats per pair
window_basic = window_with_thresh.groupby(['cmdb_id','trace_name'])['value'].agg(
    total_points_in_window='count', max_value_in_window='max'
).reset_index()

# Anomaly aggregates: count and earliest anomaly timestamp
anom_rows = window_with_thresh[window_with_thresh['is_anomaly']].copy()
if not anom_rows.empty:
    anom_agg = anom_rows.groupby(['cmdb_id','trace_name']).agg(
        anomaly_count_using_P95=('is_anomaly','sum'),
        earliest_anomaly_timestamp_in_window=('timestamp','min')
    ).reset_index()
else:
    anom_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count_using_P95','earliest_anomaly_timestamp_in_window'])

# Combine into final anomalies table
trace_anomalies = window_basic.merge(anom_agg, on=['cmdb_id','trace_name'], how='left')
trace_anomalies = trace_anomalies.merge(trace_thresholds[['cmdb_id','trace_name','P95','P5']],
                                        on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for anomaly counts with 0 and keep earliest timestamp as None where missing
trace_anomalies['anomaly_count_using_P95'] = trace_anomalies['anomaly_count_using_P95'].fillna(0).astype(int)
trace_anomalies['earliest_anomaly_timestamp_in_window'] = trace_anomalies['earliest_anomaly_timestamp_in_window'].where(
    ~trace_anomalies['earliest_anomaly_timestamp_in_window'].isna(), None
)

# 4A) thresholds table compact: cmdb_id, trace_name, P95, P5, count_all sorted by count_all desc top 50
thresholds_trace_df = trace_thresholds[['cmdb_id','trace_name','P95','P5','count_all']].sort_values(
    by='count_all', ascending=False
).head(50).reset_index(drop=True)

# 4B) anomaly candidates inside window sorted by anomaly_count desc then max_value desc, top 20
trace_anomalies_df = trace_anomalies[[
    'cmdb_id','trace_name','total_points_in_window','anomaly_count_using_P95',
    'earliest_anomaly_timestamp_in_window','max_value_in_window','P95','P5'
]].sort_values(by=['anomaly_count_using_P95','max_value_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)

# Earliest anomaly timestamp among returned trace anomalies (numeric or None)
if not trace_anomalies_df.empty and trace_anomalies_df['earliest_anomaly_timestamp_in_window'].notna().any():
    earliest_trace_anom_ts = int(trace_anomalies_df['earliest_anomaly_timestamp_in_window'].dropna().min())
else:
    earliest_trace_anom_ts = None

# Final compact outputs: component set used, thresholds table (top50), anomalies (top20), earliest anomaly ts
comp_set, thresholds_trace_df, trace_anomalies_df, earliest_trace_anom_ts

```
Out[4]:
```
```python
# Build a compact plain-English summary from cached trace analysis variables
import pytz
from datetime import datetime

# Reuse cached variables if available
try:
    comp_set
    thresholds_trace_df
    trace_anomalies_df
    earliest_trace_anom_ts
except NameError:
    # Fallbacks if any missing
    comp_set = ["IG01","IG02","Tomcat03","Tomcat04","Mysql02"]
    thresholds_trace_df = None
    trace_anomalies_df = None
    earliest_trace_anom_ts = None

tz = pytz.timezone('Asia/Shanghai')

def ts_to_str(ts):
    if ts is None:
        return 'null'
    return datetime.fromtimestamp(int(ts), tz).strftime('%Y-%m-%d %H:%M:%S%z')

def short(s, n=80):
    s = str(s)
    return s if len(s)<=n else s[:n-3]+'...'

# Counts
num_threshold_rows = int(thresholds_trace_df.shape[0]) if thresholds_trace_df is not None else 0
num_anomaly_candidates = int(trace_anomalies_df.shape[0]) if trace_anomalies_df is not None else 0

# Top examples (up to 5 each) for compact display
top_thresh_examples = []
if thresholds_trace_df is not None and not thresholds_trace_df.empty:
    for _, r in thresholds_trace_df.head(5).iterrows():
        top_thresh_examples.append({
            'cmdb_id': r['cmdb_id'],
            'trace_name': short(r['trace_name'], 80),
            'count_all': int(r['count_all']),
            'P95': float(r['P95']),
            'P5': float(r['P5'])
        })

top_anom_examples = []
if trace_anomalies_df is not None and not trace_anomalies_df.empty:
    for _, r in trace_anomalies_df.head(5).iterrows():
        top_anom_examples.append({
            'cmdb_id': r['cmdb_id'],
            'trace_name': short(r['trace_name'], 80),
            'total_points_in_window': int(r['total_points_in_window']),
            'anomaly_count_using_P95': int(r['anomaly_count_using_P95']),
            'earliest_anomaly_ts': r['earliest_anomaly_timestamp_in_window'],
            'max_value_in_window': float(r['max_value_in_window']),
            'P95': float(r['P95']),
            'P5': float(r['P5'])
        })

# Compose summary
lines = []
lines.append("Trace analysis summary (compact, plain English):")
lines.append(f"Components considered: {comp_set}")
lines.append(f"Thresholds computed on full trace series for {num_threshold_rows} (cmdb_id,trace_name) pairs.")
lines.append("Top 5 threshold examples (cmdb_id, trace_name, count_all, P95, P5):")
if top_thresh_examples:
    for t in top_thresh_examples:
        lines.append(f"  - {t['cmdb_id']}, {t['trace_name']}, count={t['count_all']}, P95={t['P95']:.6g}, P5={t['P5']:.6g}")
else:
    lines.append("  - (none)")

lines.append(f"\nAnomaly candidates inside the incident window: {num_anomaly_candidates} pairs found.")
lines.append("Top 5 anomaly candidates (cmdb_id, trace_name, total_pts, anom_cnt, earliest_ts, max_value, global_P95, global_P5):")
if top_anom_examples:
    for a in top_anom_examples:
        et = a['earliest_anomaly_ts'] if a['earliest_anomaly_ts'] is not None else 'N/A'
        et_str = ts_to_str(et) if et != 'N/A' else 'N/A'
        lines.append(
            f"  - {a['cmdb_id']}, {a['trace_name']}, pts={a['total_points_in_window']}, "
            f"anom_cnt={a['anomaly_count_using_P95']}, earliest={et}({et_str}), "
            f"max={a['max_value_in_window']:.6g}, P95={a['P95']:.6g}, P5={a['P5']:.6g}"
        )
else:
    lines.append("  - (none)")

lines.append(f"\nEarliest anomaly timestamp among returned trace anomalies: {earliest_trace_anom_ts} (UTC+8: {ts_to_str(earliest_trace_anom_ts)})")
lines.append("\nNotes: Global percentiles (P95/P5 etc.) were computed on the full trace series before window filtering; anomaly counts are within the incident window.")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(['IG01', 'IG02', 'Mysql02', 'Tomcat03', 'Tomcat04'],      cmdb_id                       trace_name          P95           P5  count_all
0       IG01         trace.self.duration_mean     0.951281     0.450591         30
1       IG01  trace.to_Tomcat01.duration_mean     0.915190     0.404699         30
2       IG01             trace.self.row_count  2196.800000  1751.500000         30
3       IG01          trace.self.duration_p95     4.350115     2.078500         30
4       IG01   trace.to_Tomcat01.duration_p95     4.283905     2.075407         30
..       ...                              ...          ...          ...        ...
45  Tomcat03          trace.to_MG01.row_count   282.300000   209.700000         30
46  Tomcat03       trace.to_MG01.duration_p95     4.261460     2.065340         30
47  Tomcat03      trace.to_MG01.duration_mean     0.904973     0.325565         30
48  Tomcat03       trace.to_MG02.duration_p95     4.032007     2.068213         30
49  Tomcat03          trace.to_MG02.row_count   288.300000   202.300000         30

[50 rows x 5 columns],      cmdb_id                   trace_name  total_points_in_window  anomaly_count_using_P95  earliest_anomaly_timestamp_in_window  max_value_in_window      P95       P5
0       IG02  trace.to_Tomcat02.row_count                      30                        3                            1615271580                282.0   278.00   200.90
1   Tomcat04      trace.to_MG02.row_count                      30                        3                            1615272060                274.0   269.00   212.55
2   Tomcat03         trace.self.row_count                      30                        2                            1615272120               4005.0  3750.30  2943.00
3   Tomcat04         trace.self.row_count                      30                        2                            1615272060               3911.0  3759.85  2933.90
4       IG02         trace.self.row_count                      30                        2                            1615272120               2248.0  2131.10  1700.90
5       IG01         trace.self.row_count                      30                        2                            1615272120               2206.0  2196.80  1751.50
6   Tomcat04      trace.to_MG01.row_count                      30                        2                            1615272060                299.0   278.35   212.70
7       IG02  trace.to_Tomcat01.row_count                      30                        2                            1615272120                295.0   279.60   208.90
8       IG02  trace.to_Tomcat04.row_count                      30                        2                            1615272060                292.0   278.25   208.30
9   Tomcat03      trace.to_MG02.row_count                      30                        2                            1615272120                292.0   288.30   202.30
10  Tomcat04    trace.from_IG02.row_count                      30                        2                            1615272060                292.0   278.25   208.30
11      IG02  trace.to_Tomcat03.row_count                      30                        2                            1615272120                291.0   280.70   206.25
12  Tomcat03    trace.from_IG02.row_count                      30                        2                            1615272120                291.0   280.70   206.25
13      IG01  trace.to_Tomcat01.row_count                      30                        2                            1615272120                289.0   284.20   207.15
14      IG01  trace.to_Tomcat03.row_count                      30                        2                            1615271580                289.0   280.10   214.70
15  Tomcat03    trace.from_IG01.row_count                      30                        2                            1615271580                289.0   280.10   214.70
16  Tomcat03      trace.to_MG01.row_count                      30                        2                            1615271580                286.0   282.30   209.70
17      IG01  trace.to_Tomcat04.row_count                      30                        2                            1615272060                283.0   274.40   215.80
18  Tomcat04    trace.from_IG01.row_count                      30                        2                            1615272060                283.0   274.40   215.80
19      IG01  trace.to_Tomcat02.row_count                      30                        2                            1615272840                279.0   277.55   215.00, 1615271580)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached log_df if available, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

# Ensure correct types
log_df['timestamp'] = log_df['timestamp'].astype(int)
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Reuse incident window timestamps
try:
    start_ts, end_ts
except NameError:
    # fallback to known values if missing
    start_ts = 1615271400
    end_ts = 1615273200

# 1) Determine component set: prefer cached filtered faults set, else fallback list
try:
    comp_set = sorted(filtered_faults_top20['cmdb_id'].dropna().unique().tolist())
    if len(comp_set) == 0:
        raise Exception("empty cached set")
except Exception:
    comp_set = ["IG01","IG02","Mysql02","Tomcat03","Tomcat04"]

# Filter logs to components of interest (full series)
logs_candidates = log_df[log_df['cmdb_id'].isin(comp_set)].copy()
logs_candidates = logs_candidates.dropna(subset=['value'])

# 1) Compute global thresholds (full series) per (cmdb_id, log_name)
agg_basic = logs_candidates.groupby(['cmdb_id','log_name'])['value'].agg(
    count_all='count', min_all='min', max_all='max', mean_all='mean'
)

quantiles = logs_candidates.groupby(['cmdb_id','log_name'])['value'].quantile(
    [0.95, 0.90, 0.50, 0.05]
).unstack(level=-1).rename(columns={0.95:'P95', 0.90:'P90', 0.50:'P50', 0.05:'P5'})

thresholds_log_df = agg_basic.join(quantiles).reset_index()

# Keep thresholds table cached (limited view will be returned)
thresholds_log_top50 = thresholds_log_df[['cmdb_id','log_name','P95','P5','count_all']].sort_values(
    by='count_all', ascending=False
).head(50).reset_index(drop=True)

# 2) Filter log rows to incident window
window_logs = logs_candidates[(logs_candidates['timestamp'] >= int(start_ts)) &
                              (logs_candidates['timestamp'] <= int(end_ts))].copy()

# Merge thresholds into window rows to evaluate anomalies
window_with_thresh = window_logs.merge(
    thresholds_log_df[['cmdb_id','log_name','P95','P5','P90']],
    on=['cmdb_id','log_name'],
    how='left'
)

# Mark anomalies: above P95 and below P5
window_with_thresh['is_anom_P95'] = window_with_thresh['value'] >= window_with_thresh['P95']
window_with_thresh['is_below_P5'] = window_with_thresh['value'] <= window_with_thresh['P5']

# Compute per-pair window aggregates
window_basic = window_with_thresh.groupby(['cmdb_id','log_name'])['value'].agg(
    total_points_in_window='count',
    max_value_in_window='max'
).reset_index()

# Anomaly aggregates for P95
anom_rows = window_with_thresh[window_with_thresh['is_anom_P95']].copy()
if not anom_rows.empty:
    anom_agg = anom_rows.groupby(['cmdb_id','log_name']).agg(
        anomaly_count_using_P95=('is_anom_P95','sum'),
        earliest_anomaly_timestamp_in_window=('timestamp','min')
    ).reset_index()
else:
    anom_agg = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count_using_P95','earliest_anomaly_timestamp_in_window'])

# Anomaly counts for P5 drops
below_rows = window_with_thresh[window_with_thresh['is_below_P5']].copy()
if not below_rows.empty:
    below_agg = below_rows.groupby(['cmdb_id','log_name']).agg(
        anomaly_count_using_P5=('is_below_P5','sum')
    ).reset_index()
else:
    below_agg = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count_using_P5'])

# Combine aggregates
log_anomalies = window_basic.merge(anom_agg, on=['cmdb_id','log_name'], how='left')
log_anomalies = log_anomalies.merge(below_agg, on=['cmdb_id','log_name'], how='left')
log_anomalies = log_anomalies.merge(thresholds_log_df[['cmdb_id','log_name','P95','P5','P90']],
                                    on=['cmdb_id','log_name'], how='left')

# Fill NaNs
log_anomalies['anomaly_count_using_P95'] = log_anomalies['anomaly_count_using_P95'].fillna(0).astype(int)
log_anomalies['anomaly_count_using_P5'] = log_anomalies['anomaly_count_using_P5'].fillna(0).astype(int)
log_anomalies['earliest_anomaly_timestamp_in_window'] = log_anomalies['earliest_anomaly_timestamp_in_window'].where(
    ~log_anomalies['earliest_anomaly_timestamp_in_window'].isna(), None
)

# 3) Prepare compact anomaly candidates table sorted by anomaly_count_using_P95 desc then max_value_in_window desc, limit top 20
log_anomalies_df = log_anomalies[[
    'cmdb_id','log_name','total_points_in_window','anomaly_count_using_P95',
    'anomaly_count_using_P5','earliest_anomaly_timestamp_in_window','max_value_in_window','P95','P5'
]].sort_values(by=['anomaly_count_using_P95','max_value_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)

# Earliest anomaly timestamp among returned log anomalies (or None)
if not log_anomalies_df.empty and log_anomalies_df['earliest_anomaly_timestamp_in_window'].notna().any():
    earliest_log_anom_ts = int(log_anomalies_df['earliest_anomaly_timestamp_in_window'].dropna().min())
else:
    earliest_log_anom_ts = None

# 4) If returned log anomalies are empty (no rows with anomaly_count_using_P95>0), compute loosened-anomaly using P90
# We'll treat "empty" as no rows where anomaly_count_using_P95 > 0
has_P95_anoms = (log_anomalies_df['anomaly_count_using_P95'].sum() > 0) if not log_anomalies_df.empty else False

loosened_log_anomalies_df = None
used_threshold = 'P95'
if not has_P95_anoms:
    # Evaluate anomalies using P90
    window_with_thresh['is_anom_P90'] = window_with_thresh['value'] >= window_with_thresh['P90']
    anom_rows_p90 = window_with_thresh[window_with_thresh['is_anom_P90']].copy()
    if not anom_rows_p90.empty:
        anom_agg_p90 = anom_rows_p90.groupby(['cmdb_id','log_name']).agg(
            anomaly_count_using_P90=('is_anom_P90','sum'),
            earliest_anomaly_timestamp_in_window_p90=('timestamp','min')
        ).reset_index()
    else:
        anom_agg_p90 = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count_using_P90','earliest_anomaly_timestamp_in_window_p90'])
    # Merge into window_basic
    loosened = window_basic.merge(anom_agg_p90, on=['cmdb_id','log_name'], how='left')
    loosened = loosened.merge(below_agg, on=['cmdb_id','log_name'], how='left')
    loosened = loosened.merge(thresholds_log_df[['cmdb_id','log_name','P90','P5']],
                              on=['cmdb_id','log_name'], how='left')
    loosened['anomaly_count_using_P90'] = loosened['anomaly_count_using_P90'].fillna(0).astype(int)
    loosened['anomaly_count_using_P5'] = loosened['anomaly_count_using_P5'].fillna(0).astype(int)
    loosened['earliest_anomaly_timestamp_in_window_p90'] = loosened['earliest_anomaly_timestamp_in_window_p90'].where(
        ~loosened['earliest_anomaly_timestamp_in_window_p90'].isna(), None
    )
    # Prepare output columns analogous to primary but indicating P90 used
    loosened_log_anomalies_df = loosened[[
        'cmdb_id','log_name','total_points_in_window','anomaly_count_using_P90',
        'anomaly_count_using_P5','earliest_anomaly_timestamp_in_window_p90','max_value_in_window','P90','P5'
    ]].rename(columns={
        'anomaly_count_using_P90':'anomaly_count_using_P95',
        'earliest_anomaly_timestamp_in_window_p90':'earliest_anomaly_timestamp_in_window',
        'P90':'P95',
    }).sort_values(by=['anomaly_count_using_P95','max_value_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)
    # earliest timestamp among loosened results
    if not loosened_log_anomalies_df.empty and loosened_log_anomalies_df['earliest_anomaly_timestamp_in_window'].notna().any():
        earliest_log_anom_ts = int(loosened_log_anomalies_df['earliest_anomaly_timestamp_in_window'].dropna().min())
    else:
        earliest_log_anom_ts = None
    # set output to loosened set and mark used threshold
    log_anomalies_df = loosened_log_anomalies_df
    thresholds_log_top50 = thresholds_log_top50  # unchanged, still global thresholds cached
    used_threshold = 'P90'

# Final compact outputs (cached thresholds and anomalies limited views)
component_set_used = comp_set
used_threshold  # indicates whether P95 or P90 was used for anomalies
component_set_used, thresholds_log_top50, log_anomalies_df, earliest_log_anom_ts, used_threshold

```
Out[5]:
```
```python
# Build a compact plain-English summary string from cached log analysis results
# (Shows numeric timestamps only as requested; no datetime conversion performed here)

# Reuse cached variables
try:
    component_set_used
    thresholds_log_top50
    log_anomalies_df
    earliest_log_anom_ts
    used_threshold
except NameError:
    # safe fallbacks
    component_set_used = ["IG01","IG02","Mysql02","Tomcat03","Tomcat04"]
    thresholds_log_top50 = pd.DataFrame(columns=['cmdb_id','log_name','P95','P5','count_all'])
    log_anomalies_df = pd.DataFrame()
    earliest_log_anom_ts = None
    used_threshold = 'P95'

# Compact counts and top examples
num_threshold_rows = int(thresholds_log_top50.shape[0])
num_log_anomalies = int(log_anomalies_df.shape[0]) if not log_anomalies_df.empty else 0

top_thresh = []
if num_threshold_rows > 0:
    for _, r in thresholds_log_top50.head(5).iterrows():
        top_thresh.append(f"{r['cmdb_id']},{r['log_name']},count={int(r['count_all'])},P95={float(r['P95']):.6g},P5={float(r['P5']):.6g}")

top_anoms = []
if num_log_anomalies > 0:
    for _, r in log_anomalies_df.head(5).iterrows():
        et = r['earliest_anomaly_timestamp_in_window'] if r['earliest_anomaly_timestamp_in_window'] is not None else 'null'
        top_anoms.append(
            f"{r['cmdb_id']},{r['log_name']},pts={int(r['total_points_in_window'])},"
            f"anom_P{('90' if used_threshold=='P90' else '95')}={int(r['anomaly_count_using_P95'])},"
            f"anom_P5={int(r['anomaly_count_using_P5'])},earliest_ts={et},max={float(r['max_value_in_window']):.6g},"
            f"P95={float(r['P95']):.6g},P5={float(r['P5']):.6g}"
        )

summary_text = (
    "Log analysis summary (compact):\n"
    f"- Components considered: {component_set_used}\n"
    f"- Thresholds computed on full log series for top {num_threshold_rows} (cmdb_id,log_name) pairs (showing up to 5 examples):\n"
)
if top_thresh:
    for t in top_thresh:
        summary_text += f"    - {t}\n"
else:
    summary_text += "    - (no threshold rows)\n"

summary_text += (
    f"- Incident window anomalies using {used_threshold} (within start_ts..end_ts): {num_log_anomalies} candidate rows (showing up to 5 examples):\n"
)
if top_anoms:
    for a in top_anoms:
        summary_text += f"    - {a}\n"
else:
    summary_text += "    - (no anomalies found using P95; P90 fallback not needed or produced none)\n"

summary_text += (
    f"- Earliest anomaly timestamp among returned log anomalies (numeric, UTC+8 assumed): {earliest_log_anom_ts}\n"
    "\nNotes:\n"
    "  * Global percentiles (P95/P5/P90) were computed on the full log series before window filtering.\n"
    "  * 'anomaly_count_using_P95' means value >= global P95; 'anomaly_count_using_P5' means value <= global P5 (possible drop).\n"
    "  * Timestamps shown are numeric epoch seconds (no timezone conversion applied here).\n"
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(['IG01', 'IG02', 'Mysql02', 'Tomcat03', 'Tomcat04'],     cmdb_id          log_name     P95      P5  count_all
0  Tomcat03   log.error_count    0.00    0.00         31
1  Tomcat03     log.row_count  563.00  434.00         31
2  Tomcat04     log.row_count  543.50  428.50         31
3  Tomcat04   log.error_count    0.00    0.00         31
4  Tomcat03  log.gc_pause_max    0.09    0.03         24
5  Tomcat04  log.gc_pause_max    0.09    0.04         24,     cmdb_id          log_name  total_points_in_window  anomaly_count_using_P95  anomaly_count_using_P5  earliest_anomaly_timestamp_in_window  max_value_in_window     P95      P5
0  Tomcat03   log.error_count                      31                       31                      31                            1615271400                  0.0    0.00    0.00
1  Tomcat04   log.error_count                      31                       31                      31                            1615271400                  0.0    0.00    0.00
2  Tomcat03  log.gc_pause_max                      24                        5                       3                            1615272000                  0.1    0.09    0.03
3  Tomcat04  log.gc_pause_max                      24                        5                       4                            1615271700                  0.1    0.09    0.04
4  Tomcat03     log.row_count                      31                        2                       2                            1615272120                572.0  563.00  434.00
5  Tomcat04     log.row_count                      31                        2                       2                            1615272060                561.0  543.50  428.50, 1615271400, 'P95')```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached globals where possible
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

# Incident window timestamps (cached)
try:
    start_ts, end_ts
except NameError:
    start_ts = 1615271400
    end_ts = 1615273200

# Components to analyze: prefer cached set if it contains Tomcat03/Tomcat04; else use specified pair
try:
    comp_set_cached = sorted(filtered_faults_top20['cmdb_id'].dropna().unique().tolist())
    # ensure we focus on Tomcat03 and Tomcat04 as requested
    if 'Tomcat03' in comp_set_cached and 'Tomcat04' in comp_set_cached:
        components = ['Tomcat03','Tomcat04']
    else:
        components = ['Tomcat03','Tomcat04']
except Exception:
    components = ['Tomcat03','Tomcat04']

# Timezone object (UTC+8) reserved (no conversions to datetime will be performed for outputs)
tz = pytz.timezone('Asia/Shanghai')

# Helper to obtain thresholds DataFrame for each source; compute fallback if missing
def ensure_metric_thresholds():
    # prefer cached thresholds_df or thresholds
    if 'thresholds_df' in globals():
        return thresholds_df[['cmdb_id','kpi_name','P95','P5']].copy()
    if 'thresholds' in globals():
        thr = thresholds.reset_index() if isinstance(thresholds, pd.DataFrame) and 'cmdb_id' not in thresholds.columns else thresholds
        return thr[['cmdb_id','kpi_name','P95','P5']].copy()
    # fallback: compute from full metric_df (should not be necessary)
    m = metric_df.copy()
    m = m.dropna(subset=['value'])
    agg = m.groupby(['cmdb_id','kpi_name'])['value'].agg(count_all='count')
    q = m.groupby(['cmdb_id','kpi_name'])['value'].quantile([0.95,0.05]).unstack(level=-1).rename(columns={0.95:'P95',0.05:'P5'})
    thr = agg.join(q).reset_index()[['cmdb_id','kpi_name','P95','P5']]
    return thr

def ensure_trace_thresholds():
    if 'trace_thresholds' in globals():
        return trace_thresholds[['cmdb_id','trace_name','P95','P5']].copy()
    # fallback compute
    t = trace_df.copy()
    t = t.dropna(subset=['value'])
    agg = t.groupby(['cmdb_id','trace_name'])['value'].agg(count_all='count')
    q = t.groupby(['cmdb_id','trace_name'])['value'].quantile([0.95,0.05]).unstack(level=-1).rename(columns={0.95:'P95',0.05:'P5'})
    thr = agg.join(q).reset_index()[['cmdb_id','trace_name','P95','P5']]
    return thr

def ensure_log_thresholds():
    if 'thresholds_log_df' in globals():
        return thresholds_log_df[['cmdb_id','log_name','P95','P5']].copy()
    if 'thresholds_log_top50' in globals():
        # thresholds_log_top50 contains subset; if that's all we have use it
        return thresholds_log_top50[['cmdb_id','log_name','P95','P5']].copy()
    # fallback compute
    l = log_df.copy()
    l = l.dropna(subset=['value'])
    agg = l.groupby(['cmdb_id','log_name'])['value'].agg(count_all='count')
    q = l.groupby(['cmdb_id','log_name'])['value'].quantile([0.95,0.05]).unstack(level=-1).rename(columns={0.95:'P95',0.05:'P5'})
    thr = agg.join(q).reset_index()[['cmdb_id','log_name','P95','P5']]
    return thr

metric_thr = ensure_metric_thresholds()
trace_thr  = ensure_trace_thresholds()
log_thr    = ensure_log_thresholds()

# Prepare windowed dataframes for fast querying (do not convert timestamps)
metric_window = metric_df[(metric_df['timestamp'] >= int(start_ts)) & (metric_df['timestamp'] <= int(end_ts))].copy()
trace_window  = trace_df[(trace_df['timestamp']  >= int(start_ts)) & (trace_df['timestamp']  <= int(end_ts))].copy()
log_window    = log_df[(log_df['timestamp']    >= int(start_ts)) & (log_df['timestamp']    <= int(end_ts))].copy()

# Normalize column names for uniform processing: use kpi_col name and thresholds key
metric_thr = metric_thr.rename(columns={'kpi_name':'kpi'})
trace_thr  = trace_thr.rename(columns={'trace_name':'kpi'})
log_thr    = log_thr.rename(columns={'log_name':'kpi'})

metric_window_for_scan = metric_window.rename(columns={'kpi_name':'kpi'})
trace_window_for_scan  = trace_window.rename(columns={'trace_name':'kpi'})
log_window_for_scan    = log_window.rename(columns={'log_name':'kpi'})

# Function to scan a source and produce breach records
def scan_source(window_df, thr_df, source_name, cm_col='cmdb_id'):
    records = []
    # restrict thresholds to components of interest
    thr_sub = thr_df[thr_df['cmdb_id'].isin(components)].copy()
    # iterate threshold rows (one per KPI)
    for _, thr in thr_sub.iterrows():
        cm = thr['cmdb_id']
        kpi = thr['kpi']
        P95 = thr['P95']
        P5  = thr['P5']
        # select window rows for this pair
        sub = window_df[(window_df['cmdb_id']==cm) & (window_df['kpi']==kpi)].copy()
        if sub.empty:
            continue
        # ensure numeric
        sub = sub.dropna(subset=['value'])
        if sub.empty:
            continue
        # determine breaches: high or low
        mask_high = sub['value'] >= P95 if pd.notna(P95) else pd.Series([False]*len(sub), index=sub.index)
        mask_low  = sub['value'] <= P5  if pd.notna(P5)  else pd.Series([False]*len(sub), index=sub.index)
        mask_any  = mask_high | mask_low
        if not mask_any.any():
            continue
        # find earliest breach timestamp
        breached = sub[mask_any].sort_values('timestamp').reset_index(drop=True)
        earliest_row = breached.iloc[0]
        earliest_ts = int(earliest_row['timestamp'])
        val_at_earliest = float(earliest_row['value'])
        # Determine breach_type: prefer "high" if value >= P95 else "low"
        if pd.notna(P95) and val_at_earliest >= P95:
            breach_type = 'high'
        elif pd.notna(P5) and val_at_earliest <= P5:
            breach_type = 'low'
        else:
            # fallback if thresholds are NaN
            breach_type = 'high' if mask_high.any() else 'low'
        # breach ratio safe calculation
        if pd.isna(P95) or P95 == 0:
            breach_ratio = float('inf') if val_at_earliest > 0 else 0.0
        else:
            breach_ratio = (val_at_earliest - P95) / P95
        # count breaches in window (union of high or low)
        count_breaches = int(mask_any.sum())
        records.append({
            'cmdb_id': cm,
            'kpi_name': kpi,
            'source': source_name,
            'earliest_breach_ts': earliest_ts,
            'value_at_earliest_breach': val_at_earliest,
            'global_P95': P95,
            'global_P5': P5,
            'breach_type': breach_type,
            'breach_ratio': breach_ratio,
            'count_breaches_in_window': count_breaches
        })
    return pd.DataFrame(records)

# Scan metric, trace, and log sources
metric_breaches = scan_source(metric_window_for_scan, metric_thr, 'metric')
trace_breaches  = scan_source(trace_window_for_scan, trace_thr, 'trace')
log_breaches    = scan_source(log_window_for_scan, log_thr, 'log')

# Combine and filter (only keep rows where at least one breach occurred - scan_source already does that)
combined = pd.concat([metric_breaches, trace_breaches, log_breaches], ignore_index=True)
if combined.empty:
    correlated_breaches_df = combined.copy()
else:
    # sort by earliest_breach_ts asc then breach_ratio desc
    correlated_breaches_df = combined.sort_values(by=['earliest_breach_ts','breach_ratio'], ascending=[True, False]).reset_index(drop=True)
    correlated_breaches_df = correlated_breaches_df.head(20).copy()

# Earliest breach overall among returned rows (numeric ts or None)
if not correlated_breaches_df.empty:
    earliest_breach_overall = int(correlated_breaches_df['earliest_breach_ts'].min())
else:
    earliest_breach_overall = None

# Keep output compact: selected columns only, top 20 as enforced
correlated_breaches_df = correlated_breaches_df[[
    'cmdb_id','kpi_name','source','earliest_breach_ts','value_at_earliest_breach',
    'global_P95','global_P5','breach_type','breach_ratio','count_breaches_in_window'
]]

# Display compact results variables (no datetime conversion)
correlated_breaches_df, earliest_breach_overall

```
Out[6]:
```
```python
# Build a compact plain-English summary of correlated anomalies for Tomcat03 and Tomcat04
import pytz
from datetime import datetime

# Reuse cached variables
try:
    correlated_breaches_df
    earliest_breach_overall
except NameError:
    correlated_breaches_df = None
    earliest_breach_overall = None

tz = pytz.timezone('Asia/Shanghai')

def ts_str(ts):
    if ts is None:
        return 'null'
    return f"{int(ts)}"

def ts_local(ts):
    if ts is None:
        return 'null'
    return datetime.fromtimestamp(int(ts), tz).strftime('%Y-%m-%d %H:%M:%S%z')

def short(s, n=80):
    s = str(s)
    return s if len(s) <= n else s[:n-3] + '...'

# Compose summary
lines = []
lines.append("Correlated anomaly timeline for Tomcat03 & Tomcat04 (compact):")

if correlated_breaches_df is None or correlated_breaches_df.empty:
    lines.append("  - No breaches found for Tomcat03 or Tomcat04 in the incident window.")
else:
    total_rows = int(correlated_breaches_df.shape[0])
    lines.append(f"  - Total breach rows found: {total_rows}")
    lines.append(f"  - Earliest breach timestamp (numeric): {ts_str(earliest_breach_overall)} (UTC+8: {ts_local(earliest_breach_overall)})")
    lines.append("")
    lines.append("  Rows (component, KPI, source, earliest_breach_ts, value_at_earliest_breach, global_P95, global_P5, breach_type, breach_ratio, count_breaches_in_window):")
    # list up to top 20 rows (already limited)
    for _, r in correlated_breaches_df.iterrows():
        lines.append(
            f"    - {r['cmdb_id']}, {short(r['kpi_name'],100)}, {r['source']}, "
            f"{int(r['earliest_breach_ts'])}, {float(r['value_at_earliest_breach']):.6g}, "
            f"P95={None if pd.isna(r['global_P95']) else float(r['global_P95']):.6g}, "
            f"P5={None if pd.isna(r['global_P5']) else float(r['global_P5']):.6g}, "
            f"type={r['breach_type']}, breach_ratio={float(r['breach_ratio']):.6g}, "
            f"count_breaches={int(r['count_breaches_in_window'])}"
        )

summary = "\n".join(lines)

# Display the compact plain-English summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  source  earliest_breach_ts  value_at_earliest_breach  global_P95  global_P5 breach_type  breach_ratio  count_breaches_in_window
0   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  metric          1615271400                  324852.0    324852.0   324852.0        high           0.0                        31
1   Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
2   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
3   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
4   Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
5   Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
6   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
7   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
8   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
9   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
10  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
11  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  metric          1615271400                     270.0       270.0      270.0        high           0.0                        31
12  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  metric          1615271400                  330351.0    330351.0   330351.0        high           0.0                        31
13  Tomcat03    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
14  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
15  Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
16  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
17  Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
18  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31
19  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...  metric          1615271400                       0.0         0.0        0.0        high           0.0                        31, 1615271400)```
```

In [ ]:
# In[7]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[7]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np

# Reuse or load dataframes
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

# Reuse incident window timestamps
try:
    start_ts, end_ts
except NameError:
    start_ts = 1615271400
    end_ts = 1615273200

# Components requested
components = ["IG01","IG02","Tomcat03","Tomcat04","Mysql02"]

# Ensure numeric types
for df in (metric_df, trace_df, log_df):
    df['timestamp'] = df['timestamp'].astype(int)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Compute global thresholds (P95, P5) on FULL series but limited to requested components for efficiency
metric_thr = (
    metric_df[metric_df['cmdb_id'].isin(components)]
    .dropna(subset=['value'])
    .groupby(['cmdb_id','kpi_name'])['value']
    .quantile([0.95,0.05])
    .unstack(level=-1)
    .rename(columns={0.95:'P95', 0.05:'P5'})
    .reset_index()
    .rename(columns={'kpi_name':'kpi'})
    [['cmdb_id','kpi','P95','P5']]
)

trace_thr = (
    trace_df[trace_df['cmdb_id'].isin(components)]
    .dropna(subset=['value'])
    .groupby(['cmdb_id','trace_name'])['value']
    .quantile([0.95,0.05])
    .unstack(level=-1)
    .rename(columns={0.95:'P95', 0.05:'P5'})
    .reset_index()
    .rename(columns={'trace_name':'kpi'})
    [['cmdb_id','kpi','P95','P5']]
)

log_thr = (
    log_df[log_df['cmdb_id'].isin(components)]
    .dropna(subset=['value'])
    .groupby(['cmdb_id','log_name'])['value']
    .quantile([0.95,0.05])
    .unstack(level=-1)
    .rename(columns={0.95:'P95', 0.05:'P5'})
    .reset_index()
    .rename(columns={'log_name':'kpi'})
    [['cmdb_id','kpi','P95','P5']]
)

# Prepare windowed data and normalize KPI column name
metric_win = metric_df[(metric_df['timestamp'] >= int(start_ts)) & (metric_df['timestamp'] <= int(end_ts)) & (metric_df['cmdb_id'].isin(components))].copy()
metric_win = metric_win.rename(columns={'kpi_name':'kpi'})[['timestamp','cmdb_id','kpi','value']]

trace_win = trace_df[(trace_df['timestamp'] >= int(start_ts)) & (trace_df['timestamp'] <= int(end_ts)) & (trace_df['cmdb_id'].isin(components))].copy()
trace_win = trace_win.rename(columns={'trace_name':'kpi'})[['timestamp','cmdb_id','kpi','value']]

log_win = log_df[(log_df['timestamp'] >= int(start_ts)) & (log_df['timestamp'] <= int(end_ts)) & (log_df['cmdb_id'].isin(components))].copy()
log_win = log_win.rename(columns={'log_name':'kpi'})[['timestamp','cmdb_id','kpi','value']]

# Helper to detect breaches and summarize per (cmdb_id,source,kpi)
def summarize_breaches(win_df, thr_df, source_name):
    if win_df.empty or thr_df.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','source','kpi','earliest_breach_ts','value_at_earliest_breach',
            'P95','P5','breach_type','count_breaches_in_window'
        ])
    merged = win_df.merge(thr_df, on=['cmdb_id','kpi'], how='left')
    # drop rows without thresholds (can't evaluate)
    merged = merged.dropna(subset=['P95','P5'], how='all').copy()
    if merged.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','source','kpi','earliest_breach_ts','value_at_earliest_breach',
            'P95','P5','breach_type','count_breaches_in_window'
        ])
    # Determine breaches
    merged['is_high'] = merged.apply(lambda r: False if pd.isna(r['P95']) else (r['value'] >= r['P95']), axis=1)
    merged['is_low']  = merged.apply(lambda r: False if pd.isna(r['P5'])  else (r['value'] <= r['P5']), axis=1)
    merged['is_breach'] = merged['is_high'] | merged['is_low']
    breaches = merged[merged['is_breach']].copy()
    if breaches.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','source','kpi','earliest_breach_ts','value_at_earliest_breach',
            'P95','P5','breach_type','count_breaches_in_window'
        ])
    # For each (cmdb_id,kpi) compute earliest breach ts and count
    grp = breaches.groupby(['cmdb_id','kpi'])
    rows = []
    for (cm,kpi), g in grp:
        earliest_ts = int(g['timestamp'].min())
        # value at earliest breach (if multiple at same ts pick max value)
        val = float(g[g['timestamp']==earliest_ts].sort_values('value', ascending=False).iloc[0]['value'])
        # corresponding P95,P5 and breach_type
        # take first row for thresholds/btype
        row0 = g[g['timestamp']==earliest_ts].sort_values('value', ascending=False).iloc[0]
        P95 = row0['P95']
        P5  = row0['P5']
        btype = 'high' if row0['is_high'] else 'low'
        count_breaches = int(g.shape[0])
        rows.append({
            'cmdb_id': cm,
            'source': source_name,
            'kpi': kpi,
            'earliest_breach_ts': earliest_ts,
            'value_at_earliest_breach': val,
            'P95': P95,
            'P5': P5,
            'breach_type': btype,
            'count_breaches_in_window': count_breaches
        })
    return pd.DataFrame(rows)

metric_br = summarize_breaches(metric_win, metric_thr, 'metric')
trace_br  = summarize_breaches(trace_win, trace_thr, 'trace')
log_br    = summarize_breaches(log_win, log_thr, 'log')

# Combine all breaches
all_br = pd.concat([metric_br, trace_br, log_br], ignore_index=True)
# For each component, find earliest breach across sources; if ties, pick one with larger breach_ratio
def safe_breach_ratio(val, P95):
    if pd.isna(P95) or P95 == 0:
        return float('inf') if (val is not None and val > 0) else 0.0
    return (val - P95) / P95

component_summary = []
for comp in components:
    subset = all_br[all_br['cmdb_id'] == comp].copy()
    if subset.empty:
        component_summary.append({
            'cmdb_id': comp,
            'earliest_breach_ts': None,
            'earliest_breach_source': None,
            'earliest_breach_kpi': None,
            'value_at_earliest_breach': None,
            'global_P95_for_that_kpi': None,
            'global_P5_for_that_kpi': None,
            'breach_type': None,
            'count_breaches_in_window_for_that_kpi': 0,
            'max_breach_ratio_across_sources_in_window': 0.0,
            'count_distinct_sources_with_breaches_in_window': 0,
            'total_breach_points_in_window': 0
        })
        continue
    # compute breach_ratio per row
    subset['breach_ratio'] = subset.apply(lambda r: safe_breach_ratio(r['value_at_earliest_breach'], r['P95']), axis=1)
    # earliest breach ts
    min_ts = int(subset['earliest_breach_ts'].min())
    # among rows with min_ts pick the one with max breach_ratio
    cand = subset[subset['earliest_breach_ts']==min_ts].sort_values('breach_ratio', ascending=False).iloc[0]
    # max_breach_ratio across all breached points for this component across sources/windows:
    # We need to compute ratio for all breached points in the window, not only earliest. Approximate by using each KPI's earliest breach ratio times count? Safer: we can compute per-source/kpi breach_ratio using value_at_earliest_breach as representative; use max across subset rows.
    max_ratio = float(subset['breach_ratio'].max())
    # count distinct sources with breaches
    distinct_sources = int(subset['source'].nunique())
    # total breach points in window: sum counts across rows (count_breaches_in_window)
    total_breach_points = int(subset['count_breaches_in_window'].sum())
    component_summary.append({
        'cmdb_id': comp,
        'earliest_breach_ts': min_ts,
        'earliest_breach_source': cand['source'],
        'earliest_breach_kpi': cand['kpi'],
        'value_at_earliest_breach': float(cand['value_at_earliest_breach']),
        'global_P95_for_that_kpi': cand['P95'],
        'global_P5_for_that_kpi': cand['P5'],
        'breach_type': cand['breach_type'],
        'count_breaches_in_window_for_that_kpi': int(cand['count_breaches_in_window']),
        'max_breach_ratio_across_sources_in_window': max_ratio,
        'count_distinct_sources_with_breaches_in_window': distinct_sources,
        'total_breach_points_in_window': total_breach_points
    })

comp_summary_df = pd.DataFrame(component_summary)

# Sort by earliest_breach_ts ascending (None last)
comp_summary_df['sort_key'] = comp_summary_df['earliest_breach_ts'].apply(lambda x: x if x is not None else 10**18)
comp_summary_df = comp_summary_df.sort_values(by=['sort_key','cmdb_id'], ascending=[True,True]).drop(columns=['sort_key']).reset_index(drop=True)

# Overall earliest breach across components (non-null)
non_null = comp_summary_df[comp_summary_df['earliest_breach_ts'].notna()].copy()
if non_null.empty:
    overall_earliest_row = pd.DataFrame([{
        'cmdb_id': None,
        'earliest_breach_ts': None,
        'earliest_breach_source': None,
        'earliest_breach_kpi': None,
        'value_at_earliest_breach': None,
        'global_P95_for_that_kpi': None,
        'breach_type': None
    }])
else:
    min_ts = int(non_null['earliest_breach_ts'].min())
    cand = non_null[non_null['earliest_breach_ts']==min_ts].sort_values('max_breach_ratio_across_sources_in_window', ascending=False).iloc[0]
    overall_earliest_row = pd.DataFrame([{
        'cmdb_id': cand['cmdb_id'],
        'earliest_breach_ts': int(cand['earliest_breach_ts']),
        'earliest_breach_source': cand['earliest_breach_source'],
        'earliest_breach_kpi': cand['earliest_breach_kpi'],
        'value_at_earliest_breach': cand['value_at_earliest_breach'],
        'global_P95_for_that_kpi': cand['global_P95_for_that_kpi'],
        'breach_type': cand['breach_type']
    }])

# Final compact outputs: comp_summary_df (5 rows) and overall_earliest_row (1 row)
comp_summary_df, overall_earliest_row

```
Out[8]:
```
```python
# Build a compact plain-English summary string from cached component summary results
import pytz
from datetime import datetime

# Reuse cached DataFrames from kernel
try:
    comp_summary_df
    overall_earliest_row
except NameError:
    raise RuntimeError("Required cached results 'comp_summary_df' or 'overall_earliest_row' not found in kernel.")

tz = pytz.timezone('Asia/Shanghai')

def fmt_ts(ts):
    if ts is None or (isinstance(ts, float) and np.isnan(ts)):
        return 'null'
    try:
        return f"{int(ts)} ({datetime.fromtimestamp(int(ts), tz).strftime('%Y-%m-%d %H:%M:%S%z')})"
    except Exception:
        return str(ts)

# Compose compact per-component lines
lines = []
lines.append("Per-component earliest-breach summary (components scanned: IG01, IG02, Tomcat03, Tomcat04, Mysql02):")
for _, r in comp_summary_df.iterrows():
    comp = r['cmdb_id']
    ets = r['earliest_breach_ts']
    if pd.isna(ets) or ets is None:
        lines.append(f"  - {comp}: no breaches in window (earliest_breach_ts=null).")
        continue
    lines.append(
        f"  - {comp}: earliest_breach_ts = {fmt_ts(ets)}, source = {r['earliest_breach_source']}, "
        f"kpi = {r['earliest_breach_kpi']}, value = {r['value_at_earliest_breach']:.6g}, "
        f"P95 = {r['global_P95_for_that_kpi']:.6g}, P5 = {r['global_P5_for_that_kpi']:.6g}, "
        f"type = {r['breach_type']}, breaches_for_kpi = {int(r['count_breaches_in_window_for_that_kpi'])}, "
        f"distinct_sources_with_breaches = {int(r['count_distinct_sources_with_breaches_in_window'])}, "
        f"total_breach_points = {int(r['total_breach_points_in_window'])}, "
        f"max_breach_ratio = {float(r['max_breach_ratio_across_sources_in_window']):.6g}"
    )

# Overall earliest breach row
oe = overall_earliest_row.iloc[0]
if oe['earliest_breach_ts'] is None or pd.isna(oe['earliest_breach_ts']):
    lines.append("\nOverall earliest breach: none detected.")
else:
    lines.append(
        "\nOverall earliest breach (component/source/kpi): "
        f"{oe['cmdb_id']}, timestamp = {fmt_ts(oe['earliest_breach_ts'])}, "
        f"source = {oe['earliest_breach_source']}, kpi = {oe['earliest_breach_kpi']}, "
        f"value = {oe['value_at_earliest_breach']:.6g}, P95 = {float(oe['global_P95_for_that_kpi']):.6g}, "
        f"type = {oe['breach_type']}"
    )

summary_text = "\n".join(lines)

# Display compact summary string (variable shown, not printed)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  earliest_breach_ts earliest_breach_source                                earliest_breach_kpi  value_at_earliest_breach  global_P95_for_that_kpi  global_P5_for_that_kpi breach_type  count_breaches_in_window_for_that_kpi  max_breach_ratio_across_sources_in_window  count_distinct_sources_with_breaches_in_window  total_breach_points_in_window
0      IG01          1615271400                 metric                    OSLinux-CPU_CPU-0_SingleCpuidle              9.878495e+01             9.866049e+01            9.787435e+01        high                                      2                                   0.272727                                               2                            928
1      IG02          1615271400                 metric           JVM-Memory_7778_JVM_Memory_HeapMemoryMax              2.040136e+09             2.040136e+09            2.040136e+09        high                                     31                                   0.818182                                               2                            877
2   Mysql02          1615271400                 metric        OSLinux-NTP_197.30.1.67_NtpServerTimeOffset              1.300000e-02             1.110000e-02           -6.000000e-03        high                                      3                                   0.949995                                               1                           2602
3  Tomcat03          1615271400                 metric                    OSLinux-CPU_CPU-3_SingleCpuidle              9.603600e+01             9.591864e+01            9.349803e+01        high                                      2                                   1.347826                                               3                           1206
4  Tomcat04          1615271400                 metric  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              3.440000e+02             3.430000e+02            3.280000e+02        high                                      7                                   0.250000                                               3                           1198,     cmdb_id  earliest_breach_ts earliest_breach_source              earliest_breach_kpi  value_at_earliest_breach  global_P95_for_that_kpi breach_type
0  Tomcat03          1615271400                 metric  OSLinux-CPU_CPU-3_SingleCpuidle                    96.036                95.918645        high)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np

# Reuse cached dataframes and variables where available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    start_ts, end_ts
except NameError:
    start_ts = 1615271400
    end_ts = 1615273200

components = ["IG01","IG02","Tomcat03","Tomcat04","Mysql02"]

# Ensure types
for df in (metric_df, trace_df, log_df):
    df['timestamp'] = df['timestamp'].astype(int)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Ensure global thresholds exist for requested components; compute if missing
def compute_thresholds(df, cm_col, kpi_col):
    thr = (
        df[df[cm_col].isin(components)]
        .dropna(subset=['value'])
        .groupby([cm_col, kpi_col])['value']
        .quantile([0.95, 0.05])
        .unstack(level=-1)
        .rename(columns={0.95:'P95', 0.05:'P5'})
        .reset_index()
        .rename(columns={cm_col:'cmdb_id', kpi_col:'kpi'})
    )
    return thr[['cmdb_id','kpi','P95','P5']]

if 'metric_thr' not in globals():
    metric_thr = compute_thresholds(metric_df, 'cmdb_id', 'kpi_name')
if 'trace_thr' not in globals():
    trace_thr = compute_thresholds(trace_df, 'cmdb_id', 'trace_name')
if 'log_thr' not in globals():
    log_thr = compute_thresholds(log_df, 'cmdb_id', 'log_name')

# Windowed data normalized to 'kpi' column
metric_win = metric_df[(metric_df['timestamp'] >= int(start_ts)) & (metric_df['timestamp'] <= int(end_ts)) & (metric_df['cmdb_id'].isin(components))].rename(columns={'kpi_name':'kpi'})[['timestamp','cmdb_id','kpi','value']]
trace_win  = trace_df[(trace_df['timestamp']  >= int(start_ts)) & (trace_df['timestamp']  <= int(end_ts)) & (trace_df['cmdb_id'].isin(components))].rename(columns={'trace_name':'kpi'})[['timestamp','cmdb_id','kpi','value']]
log_win    = log_df[(log_df['timestamp']    >= int(start_ts)) & (log_df['timestamp']    <= int(end_ts)) & (log_df['cmdb_id'].isin(components))].rename(columns={'log_name':'kpi'})[['timestamp','cmdb_id','kpi','value']]

# Helper to compute per-(cmdb_id,kpi) breach aggregates for a source
def analyze_source(window_df, thr_df, source_label):
    if window_df.empty or thr_df.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','source','kpi','total_points','count_breaches_high','count_breaches_low',
            'total_breaches','max_breach_ratio_high','earliest_breach_ts','value_at_earliest_breach','earliest_breach_type'
        ])
    merged = window_df.merge(thr_df, on=['cmdb_id','kpi'], how='left')
    merged = merged.dropna(subset=['P95','P5'], how='all').copy()
    if merged.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','source','kpi','total_points','count_breaches_high','count_breaches_low',
            'total_breaches','max_breach_ratio_high','earliest_breach_ts','value_at_earliest_breach','earliest_breach_type'
        ])
    # Flags
    merged['is_high'] = merged.apply(lambda r: False if pd.isna(r['P95']) else (r['value'] >= r['P95']), axis=1)
    merged['is_low']  = merged.apply(lambda r: False if pd.isna(r['P5'])  else (r['value'] <= r['P5']), axis=1)
    # breach ratio for high points
    def bratio(val, P95):
        if pd.isna(P95) or P95 == 0:
            return float('inf') if (not pd.isna(val) and val>0) else 0.0
        return (val - P95) / P95
    merged['bratio'] = merged.apply(lambda r: bratio(r['value'], r['P95']) if r['is_high'] else np.nan, axis=1)
    # Aggregations per KPI
    total_pts = merged.groupby(['cmdb_id','kpi'])['value'].count().rename('total_points')
    count_high = merged.groupby(['cmdb_id','kpi'])['is_high'].sum().rename('count_breaches_high')
    count_low  = merged.groupby(['cmdb_id','kpi'])['is_low'].sum().rename('count_breaches_low')
    total_b    = (count_high + count_low).rename('total_breaches')
    max_bratio = merged.groupby(['cmdb_id','kpi'])['bratio'].max().rename('max_breach_ratio_high')
    # earliest breach ts and value (any breach point)
    breaches = merged[(merged['is_high']) | (merged['is_low'])].copy()
    if breaches.empty:
        earliest = pd.DataFrame(columns=['cmdb_id','kpi','earliest_breach_ts','value_at_earliest_breach','earliest_breach_type'])
    else:
        # find earliest timestamp per pair and corresponding value and type (if both at same ts prefer high)
        breaches_sorted = breaches.sort_values(['cmdb_id','kpi','timestamp','is_high'], ascending=[True,True,True,False])
        first = breaches_sorted.groupby(['cmdb_id','kpi']).first().reset_index()[['cmdb_id','kpi','timestamp','value','is_high','is_low']]
        first = first.rename(columns={'timestamp':'earliest_breach_ts','value':'value_at_earliest_breach'})
        first['earliest_breach_type'] = first.apply(lambda r: 'high' if r['is_high'] else 'low', axis=1)
        earliest = first[['cmdb_id','kpi','earliest_breach_ts','value_at_earliest_breach','earliest_breach_type']]
    # Combine
    agg = pd.concat([total_pts, count_high, count_low, total_b, max_bratio], axis=1).reset_index()
    result = agg.merge(earliest, on=['cmdb_id','kpi'], how='left')
    # Fill NaNs appropriately
    result['count_breaches_high'] = result['count_breaches_high'].fillna(0).astype(int)
    result['count_breaches_low']  = result['count_breaches_low'].fillna(0).astype(int)
    result['total_breaches']      = result['total_breaches'].fillna(0).astype(int)
    result['max_breach_ratio_high'] = result['max_breach_ratio_high'].fillna(0.0)
    result['earliest_breach_ts'] = result['earliest_breach_ts'].where(~result['earliest_breach_ts'].isna(), None)
    result['value_at_earliest_breach'] = result['value_at_earliest_breach'].where(~result['value_at_earliest_breach'].isna(), None)
    result['earliest_breach_type'] = result['earliest_breach_type'].where(~result['earliest_breach_type'].isna(), None)
    result['source'] = source_label
    # reorder/rename columns
    result = result[[
        'cmdb_id','source','kpi','total_points','count_breaches_high','count_breaches_low',
        'total_breaches','max_breach_ratio_high','earliest_breach_ts','value_at_earliest_breach','earliest_breach_type'
    ]]
    return result

metric_summary = analyze_source(metric_win, metric_thr, 'metric')
trace_summary  = analyze_source(trace_win, trace_thr, 'trace')
log_summary    = analyze_source(log_win, log_thr, 'log')

# Combine all KPI summaries
all_kpi_summary = pd.concat([metric_summary, trace_summary, log_summary], ignore_index=True)

# Determine SEVERE KPIs per definition:
# Severe if (A) max_breach_ratio_high > 0.5 OR (B) total_breaches >= max(2, 0.5 * total_points)
def is_severe_row(r):
    condA = (r['max_breach_ratio_high'] > 0.5)
    condB = (r['total_breaches'] >= max(2, 0.5 * r['total_points']))
    return condA or condB

if not all_kpi_summary.empty:
    all_kpi_summary['is_severe'] = all_kpi_summary.apply(is_severe_row, axis=1)
else:
    all_kpi_summary['is_severe'] = pd.Series(dtype=bool)

# For severe KPIs, ensure earliest_breach_ts exists; if earliest_breach_ts is None but severe due to sustained breaches,
# we still need earliest breach timestamp among breach points: find from window rows
def fill_earliest_for_sustained(row):
    if row['earliest_breach_ts'] is not None:
        return row
    # find earliest breach timestamp from window data corresponding to this (cmdb_id,kpi,source)
    cm = row['cmdb_id']; kpi = row['kpi']; src = row['source']
    if src == 'metric':
        sub = metric_win[(metric_win['cmdb_id']==cm) & (metric_win['kpi']==kpi)]
        thr = metric_thr[(metric_thr['cmdb_id']==cm) & (metric_thr['kpi']==kpi)]
    elif src == 'trace':
        sub = trace_win[(trace_win['cmdb_id']==cm) & (trace_win['kpi']==kpi)]
        thr = trace_thr[(trace_thr['cmdb_id']==cm) & (trace_thr['kpi']==kpi)]
    else:
        sub = log_win[(log_win['cmdb_id']==cm) & (log_win['kpi']==kpi)]
        thr = log_thr[(log_thr['cmdb_id']==cm) & (log_thr['kpi']==kpi)]
    if sub.empty or thr.empty:
        row['earliest_breach_ts'] = None
        row['value_at_earliest_breach'] = None
        row['earliest_breach_type'] = None
        return row
    P95 = float(thr.iloc[0]['P95']) if not pd.isna(thr.iloc[0]['P95']) else np.nan
    P5  = float(thr.iloc[0]['P5'])  if not pd.isna(thr.iloc[0]['P5']) else np.nan
    sub = sub.dropna(subset=['value']).copy()
    sub['is_high'] = sub.apply(lambda r: False if pd.isna(P95) else (r['value'] >= P95), axis=1)
    sub['is_low']  = sub.apply(lambda r: False if pd.isna(P5) else (r['value'] <= P5), axis=1)
    breaches = sub[(sub['is_high']) | (sub['is_low'])].sort_values('timestamp')
    if breaches.empty:
        row['earliest_breach_ts'] = None
        row['value_at_earliest_breach'] = None
        row['earliest_breach_type'] = None
        return row
    earliest_ts = int(breaches.iloc[0]['timestamp'])
    val = float(breaches[breaches['timestamp']==earliest_ts].sort_values('value', ascending=False).iloc[0]['value'])
    btype = 'high' if breaches[breaches['timestamp']==earliest_ts].sort_values('value', ascending=False).iloc[0]['is_high'] else 'low'
    row['earliest_breach_ts'] = earliest_ts
    row['value_at_earliest_breach'] = val
    row['earliest_breach_type'] = btype
    return row

# Apply fill for rows where severe but earliest missing
severe_kpis = all_kpi_summary[all_kpi_summary['is_severe']].copy()
if not severe_kpis.empty:
    severe_filled = severe_kpis.apply(fill_earliest_for_sustained, axis=1)
else:
    severe_filled = severe_kpis

# For each component, find earliest severe KPI
comp_rows = []
for comp in components:
    comp_sev = severe_filled[severe_filled['cmdb_id']==comp].copy()
    if comp_sev.empty:
        comp_rows.append({
            'cmdb_id': comp,
            'earliest_severe_breach_ts': None,
            'source': None,
            'kpi_name': None,
            'value_at_earliest_severe_breach': None,
            'global_P95': None,
            'global_P5': None,
            'breach_type': None,
            'breach_ratio': 0.0,
            'count_breaches_for_this_kpi_in_window': 0,
            'total_points_for_this_kpi_in_window': 0
        })
        continue
    # remove rows with no earliest timestamp (should be filled); still handle None
    comp_sev = comp_sev[comp_sev['earliest_breach_ts'].notna()].copy()
    if comp_sev.empty:
        # all severe but no breach timestamps found (unlikely), mark nulls
        comp_rows.append({
            'cmdb_id': comp,
            'earliest_severe_breach_ts': None,
            'source': None,
            'kpi_name': None,
            'value_at_earliest_severe_breach': None,
            'global_P95': None,
            'global_P5': None,
            'breach_type': None,
            'breach_ratio': 0.0,
            'count_breaches_for_this_kpi_in_window': 0,
            'total_points_for_this_kpi_in_window': 0
        })
        continue
    # select earliest timestamp; if multiple, pick one with highest breach ratio
    comp_sev['bratio_earliest'] = comp_sev.apply(
        lambda r: float((r['value_at_earliest_breach'] - r['max_breach_ratio_high']*0 + r['max_breach_ratio_high']*0) ) , axis=1
    )  # placeholder to keep shape; we'll compute proper ratio below per row
    # compute breach_ratio for earliest value using P95
    def compute_ratio_row(r):
        P95 = r.get('P95') if 'P95' in r.index else None
        # P95 may not be in this df; fetch from appropriate thr table
        if pd.isna(P95) or P95 is None:
            # lookup
            if r['source']=='metric':
                thrrow = metric_thr[(metric_thr['cmdb_id']==r['cmdb_id']) & (metric_thr['kpi']==r['kpi'])]
            elif r['source']=='trace':
                thrrow = trace_thr[(trace_thr['cmdb_id']==r['cmdb_id']) & (trace_thr['kpi']==r['kpi'])]
            else:
                thrrow = log_thr[(log_thr['cmdb_id']==r['cmdb_id']) & (log_thr['kpi']==r['kpi'])]
            if not thrrow.empty:
                P95v = thrrow.iloc[0]['P95']
            else:
                P95v = np.nan
        else:
            P95v = P95
        val = r['value_at_earliest_breach']
        if pd.isna(P95v) or P95v == 0:
            return float('inf') if (val is not None and val>0) else 0.0
        return (val - P95v) / P95v
    comp_sev['computed_breach_ratio'] = comp_sev.apply(compute_ratio_row, axis=1)
    # choose row with minimal earliest_breach_ts then max computed_breach_ratio
    comp_sev_sorted = comp_sev.sort_values(by=['earliest_breach_ts','computed_breach_ratio'], ascending=[True,False]).reset_index(drop=True)
    chosen = comp_sev_sorted.iloc[0]
    # fetch P95 P5 for chosen KPI from appropriate thresholds table
    if chosen['source']=='metric':
        thrrow = metric_thr[(metric_thr['cmdb_id']==chosen['cmdb_id']) & (metric_thr['kpi']==chosen['kpi'])]
    elif chosen['source']=='trace':
        thrrow = trace_thr[(trace_thr['cmdb_id']==chosen['cmdb_id']) & (trace_thr['kpi']==chosen['kpi'])]
    else:
        thrrow = log_thr[(log_thr['cmdb_id']==chosen['cmdb_id']) & (log_thr['kpi']==chosen['kpi'])]
    if not thrrow.empty:
        P95v = thrrow.iloc[0]['P95']
        P5v  = thrrow.iloc[0]['P5']
    else:
        P95v = None; P5v = None
    comp_rows.append({
        'cmdb_id': chosen['cmdb_id'],
        'earliest_severe_breach_ts': int(chosen['earliest_breach_ts']),
        'source': chosen['source'],
        'kpi_name': chosen['kpi'],
        'value_at_earliest_severe_breach': float(chosen['value_at_earliest_breach']),
        'global_P95': P95v,
        'global_P5': P5v,
        'breach_type': chosen['earliest_breach_type'],
        'breach_ratio': float(chosen['computed_breach_ratio']),
        'count_breaches_for_this_kpi_in_window': int(chosen['total_breaches']),
        'total_points_for_this_kpi_in_window': int(chosen['total_points'])
    })

comp_severe_df = pd.DataFrame(comp_rows)

# Sort by earliest_severe_breach_ts ascending (None last)
comp_severe_df['sort_key'] = comp_severe_df['earliest_severe_breach_ts'].apply(lambda x: x if x is not None else 10**18)
comp_severe_df = comp_severe_df.sort_values(by=['sort_key','cmdb_id'], ascending=[True,True]).drop(columns=['sort_key']).reset_index(drop=True)

# Overall earliest severe breach: choose minimal earliest_severe_breach_ts non-null
non_null = comp_severe_df[comp_severe_df['earliest_severe_breach_ts'].notna()].copy()
if non_null.empty:
    overall_severe_row = pd.DataFrame([{
        'cmdb_id': None,
        'earliest_severe_breach_ts': None,
        'source': None,
        'kpi_name': None,
        'value_at_earliest_severe_breach': None,
        'global_P95': None,
        'global_P5': None,
        'breach_type': None,
        'breach_ratio': None,
        'count_breaches_for_this_kpi_in_window': 0,
        'total_points_for_this_kpi_in_window': 0
    }])
else:
    min_ts = int(non_null['earliest_severe_breach_ts'].min())
    cand = non_null[non_null['earliest_severe_breach_ts']==min_ts].sort_values('breach_ratio', ascending=False).iloc[0]
    overall_severe_row = pd.DataFrame([{
        'cmdb_id': cand['cmdb_id'],
        'earliest_severe_breach_ts': int(cand['earliest_severe_breach_ts']),
        'source': cand['source'],
        'kpi_name': cand['kpi_name'],
        'value_at_earliest_severe_breach': cand['value_at_earliest_severe_breach'],
        'global_P95': cand['global_P95'],
        'global_P5': cand['global_P5'],
        'breach_type': cand['breach_type'],
        'breach_ratio': cand['breach_ratio'],
        'count_breaches_for_this_kpi_in_window': int(cand['count_breaches_for_this_kpi_in_window']),
        'total_points_for_this_kpi_in_window': int(cand['total_points_for_this_kpi_in_window'])
    }])

# Final compact outputs: comp_severe_df (one row per component) and overall_severe_row (single-row DataFrame)
comp_severe_df, overall_severe_row

```
Out[9]:
```
```python
# Build a compact plain-English summary string from cached results (comp_severe_df and overall_severe_row)
# This is a summarization task requested in plain English; we display the result as a single string variable.

try:
    comp_severe_df
    overall_severe_row
except NameError:
    raise RuntimeError("Required cached results not found in kernel.")

def fmt_num(x, sig=6):
    if x is None or (isinstance(x, float) and (pd.isna(x))):
        return 'null'
    try:
        # integers (timestamps)
        if float(x).is_integer():
            return str(int(x))
    except Exception:
        pass
    # format floats with limited significant digits
    return f"{float(x):.{sig}g}"

lines = []
lines.append("Severe-breach summary per component (components: IG01, IG02, Tomcat03, Tomcat04, Mysql02).")
lines.append("Fields: earliest_severe_breach_ts (numeric), source, kpi_name, value_at_earliest_severe_breach, global_P95, global_P5, breach_type, breach_ratio, count_breaches_for_this_kpi_in_window, total_points_for_this_kpi_in_window.")
lines.append("")

for _, r in comp_severe_df.iterrows():
    comp = r['cmdb_id']
    ets = fmt_num(r['earliest_severe_breach_ts'])
    if ets == 'null':
        lines.append(f"- {comp}: no severe breaches in-window (earliest_severe_breach_ts=null).")
        continue
    lines.append(
        f"- {comp}: earliest_severe_breach_ts={ets}, source={r['source']}, "
        f"kpi={r['kpi_name']}, value={fmt_num(r['value_at_earliest_severe_breach'])}, "
        f"P95={fmt_num(r['global_P95'])}, P5={fmt_num(r['global_P5'])}, "
        f"type={r['breach_type']}, breach_ratio={fmt_num(r['breach_ratio'])}, "
        f"count_breaches={int(r['count_breaches_for_this_kpi_in_window'])}, total_points={int(r['total_points_for_this_kpi_in_window'])}"
    )

lines.append("")
# Overall earliest severe breach
oe = overall_severe_row.iloc[0]
if oe['earliest_severe_breach_ts'] is None or pd.isna(oe['earliest_severe_breach_ts']):
    lines.append("Overall earliest severe breach: none detected.")
else:
    lines.append(
        "Overall earliest severe breach: "
        f"{oe['cmdb_id']}, earliest_severe_breach_ts={fmt_num(oe['earliest_severe_breach_ts'])}, "
        f"source={oe['source']}, kpi={oe['kpi_name']}, value={fmt_num(oe['value_at_earliest_severe_breach'])}, "
        f"P95={fmt_num(oe['global_P95'])}, type={oe['breach_type']}"
    )

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  earliest_severe_breach_ts  source                                         kpi_name  value_at_earliest_severe_breach    global_P95     global_P5 breach_type  breach_ratio  count_breaches_for_this_kpi_in_window  total_points_for_this_kpi_in_window
0      IG01                 1615271400  metric         JVM-Memory_7778_JVM_Memory_HeapMemoryMax                     2.040136e+09  2.040136e+09  2.040136e+09        high      0.000000                                     62                                   31
1      IG02                 1615271400  metric         JVM-Memory_7778_JVM_Memory_HeapMemoryMax                     2.040136e+09  2.040136e+09  2.040136e+09        high      0.000000                                     62                                   31
2   Mysql02                 1615271400  metric      OSLinux-NTP_197.30.1.67_NtpServerTimeOffset                     1.300000e-02  1.110000e-02 -6.000000e-03        high      0.171171                                      3                                    3
3  Tomcat03                 1615271400  metric  OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent                     1.636690e+01  1.636690e+01  1.636690e+01        high      0.000000                                     16                                    8
4  Tomcat04                 1615271400  metric    OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                     1.000000e+00  1.000000e+00  1.000000e+00        high      0.000000                                     38                                   19,    cmdb_id  earliest_severe_breach_ts  source                                     kpi_name  value_at_earliest_severe_breach  global_P95  global_P5 breach_type  breach_ratio  count_breaches_for_this_kpi_in_window  total_points_for_this_kpi_in_window
0  Mysql02                 1615271400  metric  OSLinux-NTP_197.30.1.67_NtpServerTimeOffset                            0.013      0.0111     -0.006        high      0.171171                                      3                                    3)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached dataframes and variables where available
components = ["IG01","IG02","Tomcat03","Tomcat04","Mysql02"]

try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    start_ts, end_ts
except NameError:
    start_ts = 1615271400
    end_ts = 1615273200

# Ensure numeric types
for df in (metric_df, trace_df, log_df):
    df['timestamp'] = df['timestamp'].astype(int)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# 1) Blacklist substrings for metric KPIs (case-sensitive)
blacklist_subs = ['HeapMemoryMax','MaxThreads','SESSION','HeapMemoryMax','MaxTimeRe','_SESSION','HeapMemory']

# 2) Inclusion keywords for metric KPIs (case-sensitive substrings)
metric_keywords = ['CPU','JVM_CPULoad','SingleCpuUtil','HeapMemoryUsed','ThreadCount',
                   'Requests','Request','ErrorCount','row_count','Disk','IO','GC','gc_pause']

# Trace selection substrings
trace_keywords = ['row_count','duration','to_','from_']

# Log allowed names
log_allowed = ['log.gc_pause_max','log.row_count','log.error_count']

# 1 & 2: prepare thresholds (global) for requested components if not cached
def compute_thresholds_full(df, cm_col, kpi_col):
    thr = (
        df[df[cm_col].isin(components)]
        .dropna(subset=['value'])
        .groupby([cm_col, kpi_col])['value']
        .quantile([0.95, 0.05])
        .unstack(level=-1)
        .rename(columns={0.95:'P95', 0.05:'P5'})
        .reset_index()
        .rename(columns={cm_col:'cmdb_id', kpi_col:'kpi'})
    )
    return thr[['cmdb_id','kpi','P95','P5']]

if 'metric_thr' not in globals():
    metric_thr = compute_thresholds_full(metric_df, 'cmdb_id', 'kpi_name')
if 'trace_thr' not in globals():
    trace_thr = compute_thresholds_full(trace_df, 'cmdb_id', 'trace_name')
if 'log_thr' not in globals():
    log_thr = compute_thresholds_full(log_df, 'cmdb_id', 'log_name')

# Windowed data (incident window)
metric_win = metric_df[(metric_df['timestamp'] >= int(start_ts)) & (metric_df['timestamp'] <= int(end_ts)) & (metric_df['cmdb_id'].isin(components))].copy()
metric_win = metric_win.rename(columns={'kpi_name':'kpi'})[['timestamp','cmdb_id','kpi','value']]

trace_win = trace_df[(trace_df['timestamp'] >= int(start_ts)) & (trace_df['timestamp'] <= int(end_ts)) & (trace_df['cmdb_id'].isin(components))].copy()
trace_win = trace_win.rename(columns={'trace_name':'kpi'})[['timestamp','cmdb_id','kpi','value']]

log_win = log_df[(log_df['timestamp'] >= int(start_ts)) & (log_df['timestamp'] <= int(end_ts)) & (log_df['cmdb_id'].isin(components))].copy()
log_win = log_win.rename(columns={'log_name':'kpi'})[['timestamp','cmdb_id','kpi','value']]

# 2) Select candidate KPIs per source according to rules
# Metric: exclude blacklisted substrings and include KPIs with any metric_keywords
def metric_candidate_filter(thr_df):
    thr_sub = thr_df[thr_df['cmdb_id'].isin(components)].copy()
    def is_blacklisted(k):
        return any(sub in str(k) for sub in blacklist_subs)
    def has_keyword(k):
        return any(kw in str(k) for kw in metric_keywords)
    thr_sub['keep'] = thr_sub['kpi'].apply(lambda k: (not is_blacklisted(k)) and has_keyword(k))
    return thr_sub[thr_sub['keep']].drop(columns=['keep'])

metric_candidates = metric_candidate_filter(metric_thr)

# Trace: include those containing any trace_keywords
trace_candidates = trace_thr[trace_thr['cmdb_id'].isin(components)].copy()
trace_candidates = trace_candidates[trace_candidates['kpi'].apply(lambda k: any(kw in str(k) for kw in trace_keywords))]

# Log: restrict to allowed names and components
log_candidates = log_thr[log_thr['cmdb_id'].isin(components)].copy()
log_candidates = log_candidates[log_candidates['kpi'].isin(log_allowed)]

# Helper to safely compute breach ratio
def compute_breach_ratio(val, P95):
    if pd.isna(P95) or P95 == 0:
        return float('inf') if (val is not None and not pd.isna(val) and val > 0) else 0.0
    return (val - P95) / P95

# 3) For each candidate KPI, find earliest breach in-window and count breaches of same condition
records = []
def analyze_candidates(win_df, cand_df, source_label):
    rows = []
    for _, r in cand_df.iterrows():
        cm = r['cmdb_id']; kpi = r['kpi']; P95 = r['P95']; P5 = r['P5']
        sub = win_df[(win_df['cmdb_id']==cm) & (win_df['kpi']==kpi)].copy()
        if sub.empty:
            continue
        sub = sub.dropna(subset=['value'])
        if sub.empty:
            continue
        # determine high/low masks
        mask_high = pd.Series([False]*len(sub), index=sub.index)
        mask_low  = pd.Series([False]*len(sub), index=sub.index)
        if not pd.isna(P95):
            mask_high = sub['value'] >= P95
        if not pd.isna(P5):
            mask_low = sub['value'] <= P5
        mask_any = mask_high | mask_low
        if not mask_any.any():
            continue
        breached = sub[mask_any].sort_values('timestamp')
        earliest_row = breached.iloc[0]
        ets = int(earliest_row['timestamp'])
        val = float(earliest_row['value'])
        # determine breach_type (prefer high when both)
        if (not pd.isna(P95)) and (val >= P95):
            btype = 'high'
            count_breaches = int(mask_high.sum())
        else:
            btype = 'low'
            count_breaches = int(mask_low.sum())
        bratio = compute_breach_ratio(val, P95)
        rows.append({
            'cmdb_id': cm,
            'source': source_label,
            'kpi_name': kpi,
            'earliest_breach_ts': ets,
            'value_at_earliest_breach': val,
            'global_P95': P95,
            'global_P5': P5,
            'breach_type': btype,
            'breach_ratio': bratio,
            'count_breaches_in_window': count_breaches
        })
    return rows

records += analyze_candidates(metric_win, metric_candidates, 'metric')
records += analyze_candidates(trace_win, trace_candidates, 'trace')
records += analyze_candidates(log_win, log_candidates, 'log')

# Build DataFrame of results
results_df = pd.DataFrame(records)

# 4) Sort by earliest_breach_ts ascending and limit to top 20
if not results_df.empty:
    results_df = results_df.sort_values(by=['earliest_breach_ts','breach_ratio'], ascending=[True,False]).reset_index(drop=True)
    top20 = results_df.head(20).copy()
    overall_earliest = top20.iloc[0:1].copy()
else:
    top20 = pd.DataFrame(columns=[
        'cmdb_id','source','kpi_name','earliest_breach_ts','value_at_earliest_breach',
        'global_P95','global_P5','breach_type','breach_ratio','count_breaches_in_window'
    ])
    overall_earliest = top20.head(0)

# Keep output compact: selected columns only
top20 = top20[[
    'cmdb_id','source','kpi_name','earliest_breach_ts','value_at_earliest_breach',
    'global_P95','global_P5','breach_type','breach_ratio','count_breaches_in_window'
]]

# Display compact results variables (no datetime conversion)
top20, overall_earliest

```
Out[10]:
```
```python
# Build a compact plain-English summary using cached results (top20, overall_earliest)
# The output is textual, bounded, and uses numeric timestamps (no datetime conversion).

try:
    top20
    overall_earliest
except NameError:
    raise RuntimeError("Required cached variables 'top20' or 'overall_earliest' not found.")

def fmt_num(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return 'null'
    try:
        if float(x).is_integer():
            return str(int(x))
    except Exception:
        pass
    return f"{float(x):.6g}"

lines = []
lines.append("Selected KPI breaches (filtered & blacklisted KPIs excluded) — top entries by earliest breach timestamp:")
lines.append("Columns: cmdb_id, source, kpi_name, earliest_breach_ts (numeric), breach_type, value_at_earliest_breach, P95, P5, breach_ratio, count_breaches_in_window")
lines.append("")

if top20 is None or top20.empty:
    lines.append("  - No breaches found.")
else:
    # show up to 10 compact rows for readability (top20 already limited)
    for _, r in top20.head(10).iterrows():
        lines.append(
            f"  - {r['cmdb_id']}, {r['source']}, {r['kpi_name']}, "
            f"ts={fmt_num(r['earliest_breach_ts'])}, type={r['breach_type']}, "
            f"value={fmt_num(r['value_at_earliest_breach'])}, P95={fmt_num(r['global_P95'])}, P5={fmt_num(r['global_P5'])}, "
            f"ratio={fmt_num(r['breach_ratio'])}, count_breaches={int(r['count_breaches_in_window'])}"
        )

# Overall earliest record (first row of overall_earliest if present)
lines.append("")
if overall_earliest is None or overall_earliest.empty:
    lines.append("Overall earliest breach: none")
else:
    orow = overall_earliest.iloc[0]
    lines.append("Overall earliest breach (component/source/kpi):")
    lines.append(
        f"  - {orow['cmdb_id']}, {orow['source']}, {orow['kpi_name']}, "
        f"ts={fmt_num(orow['earliest_breach_ts'])}, type={orow['breach_type']}, "
        f"value={fmt_num(orow['value_at_earliest_breach'])}, P95={fmt_num(orow['global_P95'])}, ratio={fmt_num(orow['breach_ratio'])}, "
        f"count_breaches={int(orow['count_breaches_in_window'])}"
    )

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  source                                           kpi_name  earliest_breach_ts  value_at_earliest_breach    global_P95     global_P5 breach_type  breach_ratio  count_breaches_in_window
0   Tomcat04  metric  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...          1615271400              3.440000e+02  3.430000e+02  3.280000e+02        high      0.002915                         3
1       IG01  metric                    OSLinux-CPU_CPU-0_SingleCpuidle          1615271400              9.878495e+01  9.866049e+01  9.787435e+01        high      0.001262                         1
2   Tomcat03  metric                    OSLinux-CPU_CPU-3_SingleCpuidle          1615271400              9.603600e+01  9.591864e+01  9.349803e+01        high      0.001223                         1
3   Tomcat03  metric                    OSLinux-CPU_CPU-1_SingleCpuidle          1615271400              9.723750e+01  9.712683e+01  0.000000e+00        high      0.001139                         1
4       IG01  metric                    OSLinux-CPU_CPU-2_SingleCpuidle          1615271400              9.885191e+01  9.881403e+01  9.721620e+01        high      0.000383                         1
5       IG01  metric  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...          1615271400              3.000000e-01  3.000000e-01  3.000000e-01        high      0.000000                        31
6       IG02  metric  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...          1615271400              3.000000e-01  3.000000e-01  3.000000e-01        high      0.000000                        31
7    Mysql02  metric             Mysql-MySQL_3306_Binlog Cache Disk Use          1615271400              0.000000e+00  0.000000e+00  0.000000e+00        high      0.000000                        20
8    Mysql02  metric           Mysql-MySQL_3306_Created Tmp Disk Tables          1615271400              0.000000e+00  0.000000e+00  0.000000e+00        high      0.000000                        20
9    Mysql02  metric                 Mysql-MySQL_3306_Key Read Requests          1615271400              0.000000e+00  0.000000e+00  0.000000e+00        high      0.000000                        20
10   Mysql02  metric                Mysql-MySQL_3306_Key Write Requests          1615271400              0.000000e+00  0.000000e+00  0.000000e+00        high      0.000000                        20
11   Mysql02  metric  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...          1615271400              6.000000e-01  6.000000e-01  5.000000e-01        high      0.000000                        17
12  Tomcat03  metric  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...          1615271400              2.960000e+02  2.960000e+02  2.960000e+02        high      0.000000                        31
13  Tomcat03  metric  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...          1615271400              3.240000e+02  3.240000e+02  3.060000e+02        high      0.000000                         5
14  Tomcat04  metric  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...          1615271400              2.700000e+02  2.700000e+02  2.700000e+02        high      0.000000                        31
15  Tomcat03     log                                    log.error_count          1615271400              0.000000e+00  0.000000e+00  0.000000e+00        high      0.000000                        31
16  Tomcat04     log                                    log.error_count          1615271400              0.000000e+00  0.000000e+00  0.000000e+00        high      0.000000                        31
17  Tomcat04  metric                         OSLinux-CPU_CPU_CPUCpuUtil          1615271400              2.755120e+01  2.852757e+01  2.782223e+01         low     -0.034226                         1
18  Tomcat03  metric  Tomcat-Requests_7441-"http-nio-8003"_Processin...          1615271400              1.315739e+08  1.407660e+08  1.320052e+08         low     -0.065300                         2
19  Tomcat04  metric  Tomcat-Requests_7441-"http-nio-8003"_Processin...          1615271400              1.182652e+08  1.273362e+08  1.187100e+08         low     -0.071237                         2,     cmdb_id  source                                           kpi_name  earliest_breach_ts  value_at_earliest_breach  global_P95  global_P5 breach_type  breach_ratio  count_breaches_in_window
0  Tomcat04  metric  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...          1615271400                     344.0       343.0      328.0        high      0.002915                         3)```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached trace_df and window timestamps if available
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

try:
    start_ts, end_ts
except NameError:
    start_ts = 1615271400
    end_ts = 1615273200

components = ["IG01","IG02","Tomcat03","Tomcat04","Mysql02"]

# Ensure types
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1 & 2) Obtain global thresholds (P95,P5) computed on FULL trace series BEFORE window filtering.
# Robustly select cached variable if present; handle function name collisions.
thr = None
if 'trace_thresholds' in globals() and not callable(globals()['trace_thresholds']):
    tmp = globals()['trace_thresholds']
    if set(['cmdb_id','trace_name','P95','P5']).issubset(tmp.columns):
        thr = tmp[['cmdb_id','trace_name','P95','P5']].copy()

if thr is None and 'trace_thr' in globals():
    tmp = globals()['trace_thr']
    if 'trace_name' in tmp.columns:
        thr = tmp[['cmdb_id','trace_name','P95','P5']].copy()
    elif 'kpi' in tmp.columns:
        thr = tmp.rename(columns={'kpi':'trace_name'})[['cmdb_id','trace_name','P95','P5']].copy()

# If still None, compute on full series for requested components
if thr is None:
    thr = (
        trace_df[trace_df['cmdb_id'].isin(components)]
        .dropna(subset=['value'])
        .groupby(['cmdb_id','trace_name'])['value']
        .quantile([0.95, 0.05])
        .unstack(level=-1)
        .rename(columns={0.95:'P95', 0.05:'P5'})
        .reset_index()[['cmdb_id','trace_name','P95','P5']]
    )

# 3) Filter trace rows to the incident window and requested components
trace_win = trace_df[
    (trace_df['timestamp'] >= int(start_ts)) &
    (trace_df['timestamp'] <= int(end_ts)) &
    (trace_df['cmdb_id'].isin(components))
].copy()

# Merge global thresholds into window rows
trace_win = trace_win.merge(thr, on=['cmdb_id','trace_name'], how='left')

# Determine anomaly points where value >= global P95 (only when P95 is present)
trace_win['is_anom_P95'] = trace_win.apply(lambda r: False if pd.isna(r['P95']) else (r['value'] >= r['P95']), axis=1)

# 3) Aggregations: total points in window, anomaly count using P95, earliest anomaly ts, max value in window
total_pts = trace_win.groupby(['cmdb_id','trace_name'])['value'].count().rename('total_points_in_window').reset_index()

anom_rows = trace_win[trace_win['is_anom_P95']].copy()
if not anom_rows.empty:
    anom_agg = anom_rows.groupby(['cmdb_id','trace_name']).agg(
        anomaly_count_using_P95=('is_anom_P95','sum'),
        earliest_anomaly_timestamp_in_window=('timestamp','min'),
        max_value_in_window=('value','max')
    ).reset_index()
else:
    anom_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count_using_P95','earliest_anomaly_timestamp_in_window','max_value_in_window'])

# Combine with thresholds (P95,P5) to produce final candidates; include only pairs with anomaly_count_using_P95 > 0
candidates = total_pts.merge(anom_agg, on=['cmdb_id','trace_name'], how='left')
candidates = candidates.merge(thr, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for counts and drop pairs with zero anomalies
candidates['anomaly_count_using_P95'] = candidates['anomaly_count_using_P95'].fillna(0).astype(int)
candidates = candidates[candidates['anomaly_count_using_P95'] > 0].copy()

# 4) Sort by earliest_anomaly_timestamp_in_window ascending and limit to top 20
if not candidates.empty:
    candidates = candidates.sort_values(by=['earliest_anomaly_timestamp_in_window','max_value_in_window'], ascending=[True, False]).reset_index(drop=True)
    top20 = candidates[['cmdb_id','trace_name','total_points_in_window','anomaly_count_using_P95',
                        'earliest_anomaly_timestamp_in_window','max_value_in_window','P95','P5']].head(20).copy()
else:
    top20 = pd.DataFrame(columns=['cmdb_id','trace_name','total_points_in_window','anomaly_count_using_P95',
                                  'earliest_anomaly_timestamp_in_window','max_value_in_window','P95','P5'])

# 5) Overall earliest anomaly timestamp among returned rows (or None)
if not top20.empty and top20['earliest_anomaly_timestamp_in_window'].notna().any():
    idx = top20['earliest_anomaly_timestamp_in_window'].idxmin()
    overall_earliest = top20.loc[[idx], ['cmdb_id','trace_name','earliest_anomaly_timestamp_in_window']].reset_index(drop=True)
else:
    overall_earliest = pd.DataFrame([{'cmdb_id': None, 'trace_name': None, 'earliest_anomaly_timestamp_in_window': None}])

# Return compact textual variables (DataFrames)
top20, overall_earliest

```
Out[11]:
```
```python
# Build a compact plain-English summary from cached trace breach results
try:
    top20
    overall_earliest
except NameError:
    raise RuntimeError("Cached results 'top20' or 'overall_earliest' not found in kernel.")

def fmt_num(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return 'null'
    try:
        return str(int(x))
    except Exception:
        return str(x)

lines = []
lines.append("Trace P95 breach summary within the incident window (components: IG01, IG02, Tomcat03, Tomcat04, Mysql02).")
lines.append(f"- Rows returned (limited to top 20): {len(top20)}.")
lines.append("- Columns: cmdb_id, trace_name, total_points_in_window, anomaly_count_using_P95, earliest_anomaly_timestamp_in_window, max_value_in_window, P95, P5.")
lines.append("")
lines.append("Top breach rows (oldest first):")
# show up to first 10 rows in compact form
for _, r in top20.head(10).iterrows():
    lines.append(
        f"  - {r['cmdb_id']}, {r['trace_name']}, pts={int(r['total_points_in_window'])}, "
        f"anoms={int(r['anomaly_count_using_P95'])}, earliest_ts={fmt_num(r['earliest_anomaly_timestamp_in_window'])}, "
        f"max={float(r['max_value_in_window']):.6g}, P95={float(r['P95']):.6g}, P5={float(r['P5']):.6g}"
    )

# overall earliest anomaly among returned rows
oe = overall_earliest.iloc[0]
if oe['earliest_anomaly_timestamp_in_window'] is None:
    lines.append("\nOverall earliest anomaly: none found.")
else:
    lines.append("\nOverall earliest anomaly:")
    lines.append(
        f"  - {oe['cmdb_id']}, {oe['trace_name']}, earliest_anomaly_timestamp_in_window={fmt_num(oe['earliest_anomaly_timestamp_in_window'])}"
    )

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                       trace_name  total_points_in_window  anomaly_count_using_P95  earliest_anomaly_timestamp_in_window  max_value_in_window         P95          P5
0       IG01    trace.from_root.duration_mean                       7                        1                            1615271520             1.326600    1.238820    0.094914
1       IG01      trace.to_Tomcat03.row_count                      30                        2                            1615271580           289.000000  280.100000  214.700000
2   Tomcat03        trace.from_IG01.row_count                      30                        2                            1615271580           289.000000  280.100000  214.700000
3   Tomcat03          trace.to_MG01.row_count                      30                        2                            1615271580           286.000000  282.300000  209.700000
4       IG02      trace.to_Tomcat02.row_count                      30                        3                            1615271580           282.000000  278.000000  200.900000
5       IG02   trace.to_Tomcat02.duration_p95                      30                        2                            1615271580             5.234000    4.643915    2.096678
6       IG01   trace.to_Tomcat01.duration_p95                      30                        2                            1615271580             4.859850    4.283905    2.075407
7       IG02   trace.to_Tomcat04.duration_p95                      30                        2                            1615271580             4.484350    3.911972    2.064885
8   Tomcat04     trace.from_IG02.duration_p95                      30                        2                            1615271580             4.484350    3.911972    2.064885
9       IG01   trace.to_Tomcat02.duration_p95                      30                        2                            1615271580             4.445000    3.873727    2.026715
10      IG02   trace.to_Tomcat01.duration_p95                      30                        2                            1615271580             4.130750    4.075200    2.107883
11  Tomcat04     trace.from_root.duration_p95                       9                        1                            1615271580             2.729000    2.443760    0.257460
12      IG01     trace.from_root.duration_p95                       7                        1                            1615271580             2.683900    2.565790    0.330500
13      IG02  trace.to_Tomcat02.duration_mean                      30                        2                            1615271580             1.160610    0.984699    0.474009
14      IG01  trace.to_Tomcat02.duration_mean                      30                        2                            1615271580             1.151618    0.896142    0.387724
15      IG02  trace.to_Tomcat01.duration_mean                      30                        2                            1615271580             1.015578    0.905870    0.458755
16      IG01  trace.to_Tomcat01.duration_mean                      30                        2                            1615271580             0.986478    0.915190    0.404699
17      IG02  trace.to_Tomcat04.duration_mean                      30                        2                            1615271580             0.950350    0.910381    0.400143
18  Tomcat04    trace.from_IG02.duration_mean                      30                        2                            1615271580             0.950350    0.910381    0.400143
19  Tomcat04    trace.from_root.duration_mean                       9                        1                            1615271580             0.527133    0.434354    0.033834,   cmdb_id                     trace_name  earliest_anomaly_timestamp_in_window
0    IG01  trace.from_root.duration_mean                            1615271520)```
```